In [ ]:
from pathlib import Path

import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.datasets import STL10
import time


# DO NOT use CUDA_LAUNCH_BLOCKING=1 during normal training.
# It forces synchronization after CUDA launches and hurts throughput.
os.environ.pop("CUDA_LAUNCH_BLOCKING", None)

# Allow fast Tensor Core matmuls (TF32 on supported hardware)
torch.set_float32_matmul_precision("high")


data_root = Path(os.environ["ML_DATASETS"])

train_ds = STL10(
    root=data_root,
    split="train+unlabeled",
    download=False, #already downloaded
    transform=transforms.ToTensor(),
)

test_ds = STL10(
    root=data_root,
    split="test",
    download=False, #already downloaded
    transform=transforms.ToTensor(),
)
DATA_SEED = 44

train_generator = torch.Generator()
train_generator.manual_seed(DATA_SEED)

train_dataloader = DataLoader(
    train_ds,
    shuffle=True,
    batch_size=64,
    drop_last=True,
    generator=train_generator,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
)

test_dataloader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
)

In [ ]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_capability())
print(os.getcwd())

In [ ]:
12*28

In [ ]:
#hyperparameters

#model width and latent tokens
ae_emb_dim = 336
n_ae_latent_tokens = 64

num_blocks = 9 #dinov2 12
n_heads = 12
head_size = 28 #must be divisible by 4 for 2D rope
               #emb_dim = n_heads × head_dim <---------default ViT ratio
#z dimensions
emb_dim = 32
n_latent_tokens = 32

#Training settings
RUN_NAME = "ae_25M_selfteach_linskip" 

#learning
lr = 1e-4
beta1 = 0.9
beta2 = 0.99



#training params
crop_min = 0.2
crop_max = 0.9
encoder_patch_drop_ratio = 0.35

# 0.0 = normal RGB; 1.0 = always repeat 1 channel over others e.g. rgb -> rrr
color_drop_one_prob = 0.1

# Latent noising on the existing clean reconstruction path
clean_path_latent_noise_prob = 0.15
clean_path_latent_noise_min = 0.0
clean_path_latent_noise_max = 0.9

use_masked_view = True

branch_scale = 1e-4   # initial gain of the learned patch-IO branches (skip carries the default path)
edge_snap = 0.6       # share of small crops snapped to the image border, per axis (C4)

#z regularizers (and invariance)
ww = 0.1
swd_weight = ww  #flattened latent swd
c_swd_weight = ww   #channel wise swd
t_swd_weight = ww  #token wise swd
bwd_weight = ww   #batch wise swd for every dim

view_weight = 0.3 

patch_size = 12
batches_per_epoch = len(iter(train_dataloader))
b, c, h, w = next(iter(train_dataloader))[0].shape
pc = w//patch_size #patch columns
pr = h//patch_size #patch rows
patch_rows = pr
patch_cols = pc
total_patches = pc * pr
latent_dim = n_latent_tokens*emb_dim
patch_dim = patch_size * patch_size * c
image_dim = h*w*c

assert head_size % 4 == 0
assert head_size * n_heads == ae_emb_dim 
group_size = ae_emb_dim // emb_dim

In [ ]:
# ================= Dimension Summary =================

image_dim = c * h * w
patch_dim = c * patch_size * patch_size

ae_latent_dim = n_ae_latent_tokens * ae_emb_dim
latent_dim = n_latent_tokens * emb_dim

print("========== DIMENSION SUMMARY ==========\n")

print("IMAGE / PATCHES")
print(f"image shape:              {c} × {h} × {w}")
print(f"image dim:                {image_dim:,}")
print(f"patch size:               {patch_size} × {patch_size}")
print(f"patch grid:               {pr} × {pc}")
print(f"patch tokens:             {total_patches}")
print(f"patch dim:                {patch_dim}")
print(f"patch → AE width:         {patch_dim} → {ae_emb_dim}   "
      f"({patch_dim / ae_emb_dim:.2f}× compression)")

print("\nAE WORKSPACE")
print(f"AE token count:           {n_ae_latent_tokens}")
print(f"AE width:                 {ae_emb_dim}")
print(f"AE workspace dim:         {ae_latent_dim:,}")
print(f"attention:                {n_heads} heads × {head_size} dim "
      f"= {n_heads * head_size}")

print("\nLATENT z")
print(f"z token count:            {n_latent_tokens}")
print(f"z channel dim:            {emb_dim}")
print(f"z shape:                  ({n_latent_tokens}, {emb_dim})")
print(f"z total dim:              {latent_dim:,}")

print("\nCOMPRESSION")
print(f"patch tokens → AE tokens: {total_patches} → {n_ae_latent_tokens}   "
      f"({total_patches / n_ae_latent_tokens:.2f}×)")
print(f"AE tokens → z tokens:     {n_ae_latent_tokens} → {n_latent_tokens}   "
      f"({n_ae_latent_tokens / n_latent_tokens:.2f}×)")
print(f"AE width → z width:       {ae_emb_dim} → {emb_dim}   "
      f"({ae_emb_dim / emb_dim:.2f}×)")
print(f"AE workspace → z:         {ae_latent_dim:,} → {latent_dim:,}   "
      f"({ae_latent_dim / latent_dim:.2f}×)")
print(f"image → z:                {image_dim:,} → {latent_dim:,}   "
      f"({image_dim / latent_dim:.2f}×)")

In [ ]:
#a dataset sample
import matplotlib.pyplot as plt

image, label = train_ds[1]   # image shape: (C, H, W)

plt.imshow(image.permute(1, 2, 0))  # convert to (H, W, C)
plt.axis("off")
plt.show()

In [ ]:
print('num-batches:', len(iter(train_dataloader)), '| batch-shape:', next(iter(train_dataloader))[0].shape)

In [ ]:
import torch
from torch import nn


def apply_2d_rope(q, k, rope):
    assert q.shape == k.shape
    B, H, T, D = q.shape
    assert D % 4 == 0

    # Tables are either (P, D/4) or (B, P, D/4).
    n_patches = rope[0].shape[-2]

    # Infer the actual latent count for dense or sparse passes.
    actual_n_latent_tokens = T - n_patches
    assert actual_n_latent_tokens >= 0

    tables = []
    for table in rope:
        table = table.to(device=q.device, dtype=q.dtype)
        if table.ndim == 3:
            table = table.unsqueeze(1)  # (B, 1, P, D/4)
        tables.append(table)

    row_cos, row_sin, col_cos, col_sin = tables

    def rotate_pairs(x, cos, sin):
        x = x.reshape(B, H, n_patches, D // 4, 2)
        a, b = x.unbind(dim=-1)
        return torch.stack(
            (a * cos - b * sin, a * sin + b * cos),
            dim=-1,
        ).flatten(-2)

    def rotate_tensor(x):
        latents = x[:, :, :actual_n_latent_tokens]
        patches = x[:, :, actual_n_latent_tokens:]

        row, col = patches.chunk(2, dim=-1)
        row = rotate_pairs(row, row_cos, row_sin)
        col = rotate_pairs(col, col_cos, col_sin)

        patches = torch.cat((row, col), dim=-1)
        return torch.cat((latents, patches), dim=2)

    return rotate_tensor(q), rotate_tensor(k)


In [ ]:
#pre-post-qk-norm multi-head attention
class MHA(nn.Module):
    def __init__(self, 
                 n_heads, 
                 head_size, 
                 emb_dim, 
                 n_latent_tokens
                ):
        super().__init__()

        self.n_heads = n_heads
        self.head_size = head_size
        self.emb_dim = emb_dim
        self.n_latent_tokens = n_latent_tokens

        self.norm = nn.RMSNorm(emb_dim)
        self.QKV = nn.Linear(emb_dim, 3*n_heads * head_size)
        self.q_norm = nn.RMSNorm(head_size, eps=1e-6)
        self.k_norm = nn.RMSNorm(head_size, eps=1e-6)

        self.out_proj = nn.Linear(n_heads * head_size, emb_dim)

        self.post_norm = nn.RMSNorm(emb_dim)
        nn.init.constant_(self.post_norm.weight, 1e-4)


    def forward(self, x, rope):
        residual = x
        
        
        #useful for shape viewing/permuting below
        B, T, C = x.shape
        nh = self.n_heads
        hs = self.head_size

        #normalize
        x = self.norm(x)
        
        qkv = self.QKV(x) # (B, T, C) -> ( B, T, 3 * n_heads *head_size )

        #slice qkv aprt
        q = qkv[:, :, : nh * hs ] # ( B, T, n_heads * head_size )
        k = qkv[:, :, nh * hs : 2 * nh * hs ] # ( B, T, n_heads * head_size )
        v = qkv[:, :, 2 * nh * hs :] # ( B, T, n_heads * head_size )


        
        #reshape (B, T, n_heads*headsize) -> (B, n_heads, T, headsize)
        q = q.view(B, T, nh, hs).permute(0, 2, 1, 3)
        k = k.view(B, T, nh, hs).permute(0, 2, 1, 3)
        v = v.view(B, T, nh, hs).permute(0, 2, 1, 3)

        # After reshaping q, k to [B, heads, tokens, head_size]
        q = self.q_norm(q)
        k = self.k_norm(k)
        
        q, k = apply_2d_rope(
            q,
            k,
            rope=rope,
        )

        #q @ k.t / head_size **0.5
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=0.0,
            is_causal=False,
        )

        out = out.permute(0, 2, 1, 3).flatten(-2) # (B, T, H*hs)

        out = residual + self.post_norm(self.out_proj(out)) # (B T C)


        return out
        

In [ ]:
class MLP(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.gamma = nn.Parameter(torch.full((emb_dim,), layer_scale_gamma))
        
        self.net = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, emb_dim*4),
            nn.GELU(),
            nn.Linear(emb_dim*4, emb_dim)
        )

    def forward(self, x):
        return x + self.gamma * self.net(x)

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        hidden_dim = int(8 * emb_dim / 3)

        self.norm = nn.RMSNorm(emb_dim)

        self.gate_proj = nn.Linear(emb_dim, hidden_dim)
        self.up_proj   = nn.Linear(emb_dim, hidden_dim)
        self.down_proj = nn.Linear(hidden_dim, emb_dim)

        self.post_norm = nn.RMSNorm(emb_dim)
        nn.init.constant_(self.post_norm.weight, 1e-4)

    def forward(self, x):
        h = self.norm(x)
        h = F.silu(self.gate_proj(h)) * self.up_proj(h)
        h = self.down_proj(h)
        return x + self.post_norm(h)

In [ ]:
#transformer+MLP block
class Block(nn.Module):
    def __init__(self, 
                 n_heads, 
                 head_size, 
                 emb_dim,
                 n_latent_tokens):
        super().__init__()

        self.MHA = MHA(n_heads, head_size, emb_dim, n_latent_tokens)
        self.SwiGLU = SwiGLU(emb_dim)

    def forward(self, x, rope):

        x = self.MHA(x, rope)
        
        x = self.SwiGLU(x)

        return x
        

In [ ]:
class ChannelGroupResize(nn.Module):
    """Fixed resize of the last dim. Down: average consecutive groups. Up: repeat each channel
    over its group. Group boundaries are floor(k * hi / lo), identical in both directions."""
    def __init__(self, n_in, n_out):
        super().__init__()
        lo, hi = sorted((n_in, n_out))
        edges = torch.arange(lo + 1) * hi // lo
        sizes = edges[1:] - edges[:-1]
        self.down = n_in > n_out
        self.register_buffer("group", torch.repeat_interleave(torch.arange(lo), sizes))
        self.register_buffer("sizes", sizes.float())

    def forward(self, x):
        if not self.down:
            return x.index_select(-1, self.group)
        out = x.new_zeros(*x.shape[:-1], self.sizes.numel()).index_add_(-1, self.group, x)
        return out / self.sizes.to(x.dtype)


class LinSkipPatchEmbed(nn.Module):
    """Dense linear patchify + fixed near-lossless skip. Each patch is flattened pixel-major
    (i, j, c); the skip averages consecutive groups down to emb_dim; the learned linear sits
    behind an RMSNorm with gain branch_scale, so at init the skip is the whole path."""
    def __init__(self, patch_size, c, emb_dim, branch_scale):
        super().__init__()
        self.p = patch_size
        n = c * patch_size * patch_size
        self.skip = ChannelGroupResize(n, emb_dim)
        self.linear = nn.Linear(n, emb_dim, bias=False)
        self.norm = nn.RMSNorm(emb_dim)
        nn.init.constant_(self.norm.weight, branch_scale)

    def forward(self, x):
        B, C, H, W = x.shape
        p, rows, cols = self.p, H // self.p, W // self.p
        x = x.reshape(B, C, rows, p, cols, p).permute(0, 2, 4, 3, 5, 1).reshape(B, rows * cols, p * p * C)
        return self.skip(x) + self.norm(self.linear(x))

In [ ]:
class LinSkipUnEmbed(nn.Module):
    """Inverse of LinSkipPatchEmbed: the skip repeats each token channel over its group of the
    pixel-major (i, j, c) values; the learned linear sits behind an RMSNorm with gain branch_scale.
    Returns logits (B, c, H, W); the decoder applies the sigmoid."""
    def __init__(self, patch_size, c, emb_dim, branch_scale):
        super().__init__()
        self.p, self.c = patch_size, c
        n = c * patch_size * patch_size
        self.skip = ChannelGroupResize(emb_dim, n)
        self.linear = nn.Linear(emb_dim, n, bias=False)
        self.norm = nn.RMSNorm(n)
        nn.init.constant_(self.norm.weight, branch_scale)

    def forward(self, x, patch_rows, patch_cols):
        B = x.shape[0]
        p, c = self.p, self.c
        x = self.skip(x) + self.norm(self.linear(x))
        x = x.reshape(B, patch_rows, patch_cols, p, p, c).permute(0, 5, 1, 3, 2, 4)
        return x.reshape(B, c, patch_rows * p, patch_cols * p)

In [ ]:
class StreamRMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(emb_dim))
        self.eps = eps

    def forward(self, x):
        x = F.rms_norm(
            x,
            normalized_shape=x.shape[-2:],  # T × C together
            weight=None,
            eps=self.eps,
        )
        return x * self.weight

In [ ]:
#define core transformer model 
import math

class CoreTransformer(nn.Module):
    def __init__(self, 
                 emb_dim, 
                 num_blocks, 
                 n_heads, 
                 head_size, 
                 n_latent_tokens
                ):
        
        super().__init__()

        self.n_latent_tokens = n_latent_tokens

        self.blocks = nn.ModuleList([
            Block(n_heads, head_size, emb_dim, n_latent_tokens)
                for _ in range(num_blocks)
        ])
        
        self.norm_in = nn.RMSNorm(emb_dim)
        self.norm_out = nn.RMSNorm(emb_dim)


        #ROPE BUFFER
        pair_idx = torch.arange(head_size//4, dtype=torch.float32)
        inv_freq = 100 ** (-pair_idx / (head_size // 4))

        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.register_buffer("row_sin_cache", torch.empty(0), persistent=False)
        self.register_buffer("row_cos_cache", torch.empty(0), persistent=False)
        self.register_buffer("col_sin_cache", torch.empty(0), persistent=False)
        self.register_buffer("col_cos_cache", torch.empty(0), persistent=False)

        self.rope_cache_grid = None
 
    def forward(self, x, patch_rows, patch_cols, keep_indices=None):
        rope = self.get_rope_cache(patch_rows, patch_cols)

        if keep_indices is not None:
            rope = tuple(table[keep_indices] for table in rope)

        x = self.norm_in(x)
        for block in self.blocks:
            x = block(x, rope)

        return self.norm_out(x)

    def get_rope_cache(self, patch_rows, patch_cols):
        grid = (patch_rows, patch_cols) #current layout

        # Rebuild during training to sample fresh jitter.
        # Rebuild during eval to avoid reusing jittered training tables.
        if self.training or self.rope_cache_grid != grid:
            row_pos = (
                2 * torch.arange(
                    patch_rows, device=self.inv_freq.device, dtype=torch.float32
                ) / max(patch_rows - 1, 1) - float(patch_rows > 1)
            ).repeat_interleave(patch_cols)
            
            col_pos = (
                2 * torch.arange(
                    patch_cols, device=self.inv_freq.device, dtype=torch.float32
                ) / max(patch_cols - 1, 1) - float(patch_cols > 1)
            ).repeat(patch_rows)
            
    
            scale = 1.0

            if self.training:
                # Uniform in log-space: shrinking and enlarging are balanced.
                log_scale = torch.empty(
                    (), device=self.inv_freq.device
                ).uniform_(-math.log(2), math.log(2))
            
                scale = log_scale.exp()
            
            row_angles = (
                2 * math.pi * scale
                * row_pos[:, None] * self.inv_freq[None, :]
            )
            col_angles = (
                2 * math.pi * scale
                * col_pos[:, None] * self.inv_freq[None, :]
            )
    
            self.row_cos_cache = row_angles.cos()
            self.row_sin_cache = row_angles.sin()
            self.col_cos_cache = col_angles.cos()
            self.col_sin_cache = col_angles.sin()
    
            # Training tables must not be reused during evaluation.
            self.rope_cache_grid = None if self.training else grid
        return (
            self.row_cos_cache,
            self.row_sin_cache,
            self.col_cos_cache,
            self.col_sin_cache,
        )

In [ ]:
class SparseMiddleTransformer(CoreTransformer):
    """
    Shallow blocks → sparse middle blocks → fusion → final blocks.

    Uses the existing CoreTransformer's RoPE setup, including jitter.
    Input-masked patches remain absent throughout this core.
    """

    def __init__(
        self,
        emb_dim,
        num_blocks,
        n_heads,
        head_size,
        n_latent_tokens,
        keep_ratio=0.25,
        shallow_blocks=1,
        final_blocks=1,
    ):
        super().__init__(
            emb_dim=emb_dim,
            num_blocks=num_blocks,
            n_heads=n_heads,
            head_size=head_size,
            n_latent_tokens=n_latent_tokens,
        )

        if not 0 < keep_ratio <= 1:
            raise ValueError("keep_ratio must be in (0, 1].")
        if shallow_blocks < 1 or final_blocks < 1:
            raise ValueError("Need at least one shallow and final block.")
        if shallow_blocks + final_blocks >= num_blocks:
            raise ValueError("Need at least one middle block.")

        self.keep_ratio = keep_ratio
        self.shallow_blocks = shallow_blocks
        self.middle_end = num_blocks - final_blocks

        # Set False for full-token fine-tuning.
        self.sparse_middle = True

        # Fuse shallow and deep features: 2C → C.
        self.fusion = nn.Linear(2 * emb_dim, emb_dim)

        # Start with equal contributions from both paths.
        with torch.no_grad():
            eye = torch.eye(emb_dim)
            self.fusion.weight.copy_(
                torch.cat((0.5 * eye, 0.5 * eye), dim=1)
            )
            self.fusion.bias.zero_()

    @staticmethod
    def gather_tokens(x, indices):
        return x.gather(
            dim=1,
            index=indices.unsqueeze(-1).expand(-1, -1, x.shape[-1]),
        )

    @staticmethod
    def gather_rope(rope, indices):
        selected = []
        for table in rope:
            if table.ndim == 2:
                # Shared full-grid table.
                selected.append(table[indices])
            else:
                # Already selected per image by input masking.
                selected.append(
                    table.gather(
                        dim=1,
                        index=indices.unsqueeze(-1).expand(
                            -1, -1, table.shape[-1],
                        ),
                    )
                )
        return tuple(selected)

    @staticmethod
    def sample_indices(batch, count, keep, device):
        # Sort selected indices to preserve original token order.
        return (
            torch.rand(batch, count, device=device)
            .topk(keep, dim=1, largest=False)
            .indices.sort(dim=1)
            .values
        )

    def forward(self, x, patch_rows, patch_cols, keep_indices=None):
        
        x = self.norm_in(x)

        
        rope = self.get_rope_cache(patch_rows, patch_cols)
        
        # Input masking: x already contains only visible patches.
        if keep_indices is not None:
            rope = self.gather_rope(rope, keep_indices)


        B, T, C = x.shape

        # RoPE tells us exactly how many spatial patch tokens are present.
        P = rope[0].shape[-2]
        
        # Everything before those final patch tokens is latent memory.
        L = T - P
        
        assert L >= 0, (
            f"Sequence has {T} total tokens but RoPE contains "
            f"{P} patch positions."
        )

        # Shallow path sees all available tokens.
        for i in range(self.shallow_blocks):
            x = self.blocks[i](x, rope)

        shallow = x

        use_sparse = (
            self.training
            and self.sparse_middle
            and self.keep_ratio < 1
        )

        if use_sparse:
            # Fixed counts. At least one survives in each nonempty group.
            keep_latents = min(L, max(1, int(L * self.keep_ratio)))
            keep_patches = min(P, max(1, int(P * self.keep_ratio)))

            latent_ids = self.sample_indices(
                B, L, keep_latents, x.device,
            )
            patch_ids = self.sample_indices(
                B, P, keep_patches, x.device,
            )

            # Latents remain the prefix; patches follow.
            token_ids = torch.cat(
                (latent_ids, L + patch_ids), dim=1,
            )

            deep = self.gather_tokens(shallow, token_ids)
            middle_rope = self.gather_rope(rope, patch_ids)
        else:
            deep = shallow
            middle_rope = rope

        for i in range(self.shallow_blocks, self.middle_end):
            deep = self.blocks[i](deep, middle_rope)

        if use_sparse:
            # Restore deep features to their original positions.
            deep = torch.zeros_like(shallow).scatter(
                dim=1,
                index=token_ids.unsqueeze(-1).expand(-1, -1, C),
                src=deep,
            )

        x = self.fusion(torch.cat((shallow, deep), dim=-1))

        for i in range(self.middle_end, len(self.blocks)):
            x = self.blocks[i](x, rope)

        return self.norm_out(x)

In [ ]:
#define encoder model 
    
class Encoder(nn.Module):
    def __init__(self, 
                 patch_size, 
                 in_channels, 
                 ae_emb_dim,
                 emb_dim, 
                 n_ae_latent_tokens,
                 n_latent_tokens,
                 num_blocks, 
                 n_heads, 
                 head_size, 
                ):
        
        super().__init__()


        self.ae_emb_dim = ae_emb_dim
        self.emb_dim = emb_dim

        self.n_latent_tokens = n_latent_tokens
        self.n_ae_latent_tokens = n_ae_latent_tokens

        self.latent_tokens = nn.Parameter(
            torch.randn(1, n_ae_latent_tokens, ae_emb_dim) * 0.02
        )

        assert patch_size == 12 and in_channels == 3
        self.embed = LinSkipPatchEmbed(patch_size, in_channels, ae_emb_dim, branch_scale)

        self.core_transformer = CoreTransformer(emb_dim = ae_emb_dim, 
                                                 num_blocks = num_blocks, 
                                                 n_heads = n_heads, 
                                                 head_size = head_size, 
                                                 n_latent_tokens = n_ae_latent_tokens
                                                )

        self.emb_out_proj = nn.Linear(ae_emb_dim, emb_dim, bias=False)
        nn.init.normal_(self.emb_out_proj.weight, std=branch_scale)   # z skip carries the default path

        self.z_group_size = ae_emb_dim // emb_dim

    def forward(self, image, keep_indices=None):
        
        patch_rows = image.shape[-2] // 12
        patch_cols = image.shape[-1] // 12
        patches = self.embed(image)

        if keep_indices is not None:
            indices = keep_indices.unsqueeze(-1).expand(
                -1, -1, patches.shape[-1],
            )
            patches = patches.gather(dim=1, index=indices)
        
        # cat latent tokens with patch embed
        B = patches.shape[0]
        
        latents = self.latent_tokens.expand(B, -1, -1)
        x = torch.cat([latents, patches], dim=1)
        
        x = self.core_transformer(
            x, patch_rows, patch_cols,
            keep_indices=keep_indices,
        )

        latents = x[:, :self.n_latent_tokens]
        
        B, T, D = latents.shape
        shortcut = F.adaptive_avg_pool1d(
            latents.reshape(B * T, 1, D), self.emb_dim,
        ).reshape(B, T, self.emb_dim)
        z = self.emb_out_proj(latents) + shortcut
        
        return z

In [ ]:
# define decoder model

class Decoder(nn.Module):
    def __init__(
        self,
        patch_size,
        out_channels,
        ae_emb_dim,
        emb_dim,
        n_ae_latent_tokens,
        n_latent_tokens,
        num_blocks,
        n_heads,
        head_size,
    ):
        super().__init__()

        self.ae_emb_dim = ae_emb_dim
        self.emb_dim = emb_dim

        self.n_ae_latent_tokens = n_ae_latent_tokens
        self.n_latent_tokens = n_latent_tokens

        self.latent_slots = nn.Parameter(
            torch.randn(1, n_ae_latent_tokens, ae_emb_dim) * 0.02
        )

        self.emb_in_proj = nn.Linear(
            emb_dim,
            ae_emb_dim,
            bias=False,
        )
        nn.init.normal_(self.emb_in_proj.weight, std=branch_scale)    # z skip carries the default path

        self.learned_patch_token = nn.Parameter(
            torch.randn(1, 1, ae_emb_dim) * 0.02
        )

        self.core_transformer = CoreTransformer(
            emb_dim=ae_emb_dim,
            num_blocks=num_blocks,
            n_heads=n_heads,
            head_size=head_size,
            n_latent_tokens=n_ae_latent_tokens,
        )

        assert patch_size == 12 and out_channels == 3
        self.un_embed = LinSkipUnEmbed(patch_size, out_channels, ae_emb_dim, branch_scale)

        self.z_group_size = ae_emb_dim // emb_dim

        
    def forward(self, z, latent_keep_indices=None):
    
        # (B, n_z_tokens, emb_dim) -> (B, n_z_tokens, ae_emb_dim)
        B, T, Cz = z.shape
        shortcut = F.interpolate(
            z.reshape(B * T, 1, Cz), size=self.ae_emb_dim, mode="nearest",
        ).reshape(B, T, self.ae_emb_dim)
        latents = self.emb_in_proj(z) + shortcut
        n_z_tokens = latents.shape[1]
        
        # (B, n_z_tokens, emb_dim) -> (B, n_ae_latent_tokens, ae_emb_dim)
        latents = F.pad(
            latents,
            (
                0, 0,
                0, self.n_ae_latent_tokens - n_z_tokens,
            ),
        )
        
        latents = latents + self.latent_slots
        
        # Physically drop masked z tokens.
        # Extra decoder latent slots remain untouched.
        if latent_keep_indices is not None:
            kept_latents = latents[:, :n_z_tokens].gather(
                dim=1,
                index=latent_keep_indices.unsqueeze(-1).expand(
                    -1, -1, latents.shape[-1]
                ),
            )
        
            latents = torch.cat(
                [
                    kept_latents,
                    latents[:, n_z_tokens:],
                ],
                dim=1,
            )
    
        B, T, C = latents.shape
    
        n_patches = patch_rows * patch_cols
    
        patches = self.learned_patch_token.expand(
            B,
            n_patches,
            -1,
        )
    
        x = torch.cat([latents, patches], dim=1)
    
        x = self.core_transformer(
            x,
            patch_rows,
            patch_cols,
        )

    
        # Patch tokens are always the final n_patches tokens
        patch_embeddings = x[:, -n_patches:]
    
        images = torch.sigmoid(self.un_embed(patch_embeddings, patch_rows, patch_cols))
    
        return images

        

In [ ]:
def sliced_wasserstein_distance(
    z,
    prior,
    n_slices=1024,
    bulk_sigma=1.0,       # smaller = stronger tail discount; None = unweighted
):
    assert z.shape == prior.shape
    if bulk_sigma is not None and bulk_sigma <= 0:
        raise ValueError("bulk_sigma must be positive or None")

    z = z.flatten(1)                         # (B, D)
    prior = prior.detach().flatten(1)        # fixed target
    B, D = z.shape

    slices = F.normalize(
        torch.randn(D, n_slices, device=z.device, dtype=z.dtype),
        dim=0,
    )                                       # (D, S)

    z_slices = torch.sort(z @ slices, dim=0).values.float()
    p_slices = torch.sort(prior @ slices, dim=0).values.float()

    squared_error = (z_slices - p_slices).square()

    if bulk_sigma is None:
        loss = squared_error.mean()
    else:
        log_weights = -0.5 * (p_slices / bulk_sigma).square()

        # Normalize across batch ranks, separately for each slice.
        # Softmax stably computes exp(log_weights) / sum(exp(log_weights)).
        weights = torch.softmax(log_weights, dim=0)

        loss = (weights * squared_error).sum(dim=0).mean()

     
    return loss

In [ ]:
def basis_wasserstein_distance(z, prior, bulk_sigma=1.0):
    assert z.shape == prior.shape
    assert z.ndim == 3, "Expected z shaped (B, T, C)"

    if bulk_sigma is not None and bulk_sigma <= 0:
        raise ValueError("bulk_sigma must be positive or None")

    prior = prior.detach()

    def axis_loss(source, target, dim):
        source = torch.sort(source, dim=dim).values.float()
        target = torch.sort(target, dim=dim).values.float()

        squared_error = (source - target).square()

        if bulk_sigma is None:
            return squared_error.mean()

        log_weights = -0.5 * (target / bulk_sigma).square()
        weights = torch.softmax(log_weights, dim=dim)

        return (weights * squared_error).sum(dim=dim).mean()

    loss_b = axis_loss(z, prior, dim=0)   # across images, per (t, c)
    loss_t = axis_loss(z, prior, dim=1)   # across tokens, per (b, c)
    loss_c = axis_loss(z, prior, dim=2)   # across channels, per (b, t)
    loss_e = axis_loss(
        z.flatten(1), prior.flatten(1), dim=1
    )                                    # across T*C, per image

    loss = loss_b # + 0.1 * (loss_t + loss_c + loss_e) / 3
    
 
    return loss

In [ ]:
def token_sliced_wasserstein_distance(
    z,
    prior,
    n_slices=32,
    bulk_sigma=1.0,
):
    """
    Token-space SWD.

    z/prior: (B, T, C)

    Reshapes to (B*C, T), so each channel across the token sequence
    is one sample, and random slices operate across the T token dimensions.

    This directly pressures token combinations/correlations
    toward the isotropic Gaussian prior.
    """
    assert z.shape == prior.shape
    assert z.ndim == 3, "Expected z shaped (B, T, C)"

    if bulk_sigma is not None and bulk_sigma <= 0:
        raise ValueError("bulk_sigma must be positive or None")

    B, T, C = z.shape

    # Each channel's values across T tokens becomes one T-dimensional observation.
    z = z.permute(0, 2, 1).reshape(B * C, T)
    prior = prior.detach().permute(0, 2, 1).reshape(B * C, T)

    # Random directions through token space.
    slices = F.normalize(
        torch.randn(
            T,
            n_slices,
            device=z.device,
            dtype=z.dtype,
        ),
        dim=0,
    )  # (T, S)

    # Project each observation onto random token-space directions.
    z_slices = torch.sort(
        z @ slices,
        dim=0,
    ).values.float()

    p_slices = torch.sort(
        prior @ slices,
        dim=0,
    ).values.float()

    squared_error = (z_slices - p_slices).square()

    if bulk_sigma is None:
        loss = squared_error.mean()

    else:
        log_weights = -0.5 * (p_slices / bulk_sigma).square()

        # Normalize across the B*C observations for each slice.
        weights = torch.softmax(
            log_weights,
            dim=0,
        )

        loss = (
            weights * squared_error
        ).sum(dim=0).mean()

    return loss 

In [ ]:
def channel_sliced_wasserstein_distance(
    z,
    prior,
    n_slices=32,
    bulk_sigma=1.0,
):
    """
    Channel-space SWD.

    z/prior: (B, T, C)

    Reshapes to (B*T, C), so each token embedding is one sample
    and random slices operate across the C channel dimensions.

    This directly pressures channel combinations/correlations
    toward the isotropic Gaussian prior.
    """
    assert z.shape == prior.shape
    assert z.ndim == 3, "Expected z shaped (B, T, C)"

    if bulk_sigma is not None and bulk_sigma <= 0:
        raise ValueError("bulk_sigma must be positive or None")

    B, T, C = z.shape

    # Each token embedding becomes one C-dimensional observation.
    z = z.reshape(B * T, C)
    prior = prior.detach().reshape(B * T, C)

    # Random directions through channel space.
    slices = F.normalize(
        torch.randn(
            C,
            n_slices,
            device=z.device,
            dtype=z.dtype,
        ),
        dim=0,
    )  # (C, S)

    # Project each token embedding onto each random channel direction.
    z_slices = torch.sort(
        z @ slices,
        dim=0,
    ).values.float()

    p_slices = torch.sort(
        prior @ slices,
        dim=0,
    ).values.float()

    squared_error = (z_slices - p_slices).square()

    if bulk_sigma is None:
        loss = squared_error.mean()

    else:
        log_weights = -0.5 * (p_slices / bulk_sigma).square()

        # Normalize across the B*T observations for each slice.
        weights = torch.softmax(
            log_weights,
            dim=0,
        )

        loss = (
            weights * squared_error
        ).sum(dim=0).mean()

     
    return loss

In [ ]:
def laplacian(x):
    C = x.shape[1]

    kernel = torch.tensor(
        [[0, -1,  0],
         [-1, 4, -1],
         [0, -1,  0]],
        device=x.device,
        dtype=x.dtype,
    )

    kernel = kernel[None, None].repeat(C, 1, 1, 1)

    return F.conv2d(x, kernel, padding=1, groups=C)

In [ ]:
@torch.no_grad()
def render_reconstruction(masked=False, show=True, training_mode=False, split='train'):
    modes = [
        (module, module.training)
        for model in (encoder, decoder)
        for module in model.modules()
    ]

    try:
        encoder.train(training_mode)
        decoder.train(training_mode)

        if split == 'train':
            features, _ = next(iter(train_dataloader))
        else:
            features, _ = next(iter(test_dataloader))
        
        features = features[:8].to(device)

        if masked:
            mask, keep_indices = make_mask(
                features,
                mask_ratio=encoder_patch_drop_ratio,
                return_keep_indices=True,
            )
            z = encoder(features, keep_indices=keep_indices)
            pixel_mask = make_pixel_mask(mask, features)
            display_features = features * (1.0 - pixel_mask)
        else:
            z = encoder(features)
            display_features = features

        x_hat = decoder(z)

        n = features.shape[0]
        fig, axes = plt.subplots(
            2, n, figsize=(2 * n, 4), squeeze=False,
        )

        for i in range(n):
            axes[0, i].imshow(
                display_features[i]
                .float().cpu().permute(1, 2, 0).clamp(0, 1)
            )
            axes[0, i].axis("off")

            axes[1, i].imshow(
                x_hat[i]
                .float().cpu().permute(1, 2, 0).clamp(0, 1)
            )
            axes[1, i].axis("off")

        axes[0, 0].set_ylabel(
            "masked input" if masked else "input",
            rotation=0, labelpad=45, va="center",
        )
        axes[1, 0].set_ylabel(
            "recon",
            rotation=0, labelpad=35, va="center",
        )

        plt.tight_layout()

        if show:
            plt.show()
        else:
            plt.close(fig)

        return fig

    finally:
        for module, training in modes:
            module.training = training

In [ ]:
#helper: def reconstruct images
@torch.no_grad()
def reconstruct_images(images, close_fig = False):
    
    x_hat = decoder(encoder(images))

    x = images[0].detach().cpu().permute(1, 2, 0)
    recon = x_hat[0].detach().cpu().permute(1, 2, 0)
    #recon = x_hat[0,0]

    fig, axs = plt.subplots(1, 2, figsize = (6,3))

    axs[0].imshow(x)
    axs[0].set_title('input')
    axs[0].axis('off')

    
    axs[1].imshow(recon)
    axs[1].set_title('reconstruction')
    axs[1].axis('off')

    if close_fig:
        plt.close(fig)

In [ ]:
#helper: def plot_training_snapshot
import os
import torch
import matplotlib.pyplot as plt


@torch.no_grad()
def plot_training_snapshot(
    recon_targets,
    step,
    total_steps,
    save_path=None,
    show=False,
):
    """
    Plot four fixed reconstruction examples.

    Row 1: original targets
    Row 2: model reconstructions

    Assumes:
        recon_targets: (4, C, H, W)
        encoder and decoder exist in the outer scope
    """
    if recon_targets.ndim != 4:
        raise ValueError(
            f"Expected recon_targets shaped (4, C, H, W), "
            f"got {recon_targets.shape}"
        )

    if recon_targets.shape[0] != 4:
        raise ValueError(
            f"Expected exactly 4 reconstruction targets, "
            f"got {recon_targets.shape[0]}"
        )

    # Infer the model device rather than relying on a global `device`.
    model_device = next(encoder.parameters()).device
    targets = recon_targets.to(model_device)

    encoder_was_training = encoder.training
    decoder_was_training = decoder.training

    encoder.eval()
    decoder.eval()

    try:
        reconstructions = decoder(encoder(targets))
    finally:
        encoder.train(encoder_was_training)
        decoder.train(decoder_was_training)

    targets = targets.detach().float().cpu()
    reconstructions = reconstructions.detach().float().cpu()

    if reconstructions.shape != targets.shape:
        raise ValueError(
            "Target and reconstruction shapes do not match:\n"
            f"targets:         {targets.shape}\n"
            f"reconstructions: {reconstructions.shape}"
        )

    def prepare_for_imshow(image):
        """
        Convert:
            (1, H, W) -> (H, W)
            (3, H, W) -> (H, W, 3)
        """
        if image.ndim != 3:
            raise ValueError(
                f"Expected individual image shaped (C, H, W), got {image.shape}"
            )

        channels = image.shape[0]

        if channels == 1:
            return image.squeeze(0).clamp(0, 1).numpy()

        if channels in (3, 4):
            return image.permute(1, 2, 0).clamp(0, 1).numpy()

        raise ValueError(f"Cannot display image with {channels} channels")

    # Per-example reconstruction error for a compact progress signal.
    mse = (
        (targets - reconstructions)
        .square()
        .flatten(start_dim=1)
        .mean(dim=1)
    )

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(12, 6),
        gridspec_kw={
            "hspace": 0.08,
            "wspace": 0.08,
        },
    )

    fig.suptitle(
        f"Autoencoder Training — Step {step:,} / {total_steps:,}",
        fontsize=15,
        y=0.97,
    )

    for i in range(4):
        target = prepare_for_imshow(targets[i])
        reconstruction = prepare_for_imshow(reconstructions[i])

        # Top row: original
        axes[0, i].imshow(
            target,
            cmap="gray" if targets.shape[1] == 1 else None,
            vmin=0,
            vmax=1,
            interpolation="nearest",
        )
        axes[0, i].set_title(
            f"Sample {i + 1}\nMSE: {mse[i].item():.4f}",
            fontsize=10,
        )
        axes[0, i].axis("off")

        # Bottom row: reconstruction
        axes[1, i].imshow(
            reconstruction,
            cmap="gray" if reconstructions.shape[1] == 1 else None,
            vmin=0,
            vmax=1,
            interpolation="nearest",
        )
        axes[1, i].axis("off")

    # Shared row labels avoid repeating eight titles.
    fig.text(
        0.025,
        0.69,
        "Target",
        rotation=90,
        va="center",
        ha="center",
        fontsize=12,
    )
    fig.text(
        0.025,
        0.29,
        "Reconstruction",
        rotation=90,
        va="center",
        ha="center",
        fontsize=12,
    )

    fig.subplots_adjust(
        top=0.86,
        bottom=0.04,
        left=0.06,
        right=0.99,
    )

    if save_path is not None:
        save_dir = os.path.dirname(save_path)

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)

        fig.savefig(
            save_path,
            dpi=120,
            bbox_inches="tight",
        )

    if show:
        plt.show()

    plt.close(fig)

In [ ]:
# get split losses

@torch.inference_mode()
def get_split_loss(split):
    encoder_was_training = encoder.training
    decoder_was_training = decoder.training

    encoder.eval()
    decoder.eval()

    if split == "test":
        dataloader = test_dataloader
        max_batches = None

    elif split == "train":
        dataloader = train_dataloader
        max_batches = 20

    else:
        raise ValueError(f"Unknown split: {split}")

    masked_squared_error = torch.zeros((), device=device)
    clean_squared_error = torch.zeros((), device=device)
    num_elements = 0

    try:
        for batch_index, (x, _) in enumerate(dataloader):
            if max_batches is not None and batch_index >= max_batches:
                break

            x = x.to(device)

            x_hat_masked = decoder(
                encoder(x, make_mask(x))
            )

            masked_squared_error += F.mse_loss(
                x_hat_masked,
                x,
                reduction="sum",
            )

            x_hat_clean = decoder(
                encoder(x)
            )

            clean_squared_error += F.mse_loss(
                x_hat_clean,
                x,
                reduction="sum",
            )

            num_elements += x.numel()

    finally:
        encoder.train(encoder_was_training)
        decoder.train(decoder_was_training)

    loss_masked = masked_squared_error / num_elements
    loss_clean = clean_squared_error / num_elements

    return loss_masked, loss_clean

In [ ]:
#log metrics
from collections import defaultdict
from copy import deepcopy

if "logs" not in globals():
    logs = defaultdict(list)


def clear_logs():
    logs.clear()


def log_metrics(**metrics):
    for name, value in metrics.items():
        if value is None:
            continue

        if torch.is_tensor(value):
            value = value.detach().item()

        logs[name].append(value)

def collect_metrics(metrics):
    names = []
    tensors = []
    numbers = {}

    for name, value in metrics.items():
        if value is None:
            continue

        if torch.is_tensor(value):
            names.append(name)
            tensors.append(value.detach().reshape(()))
        else:
            numbers[name] = value

    if tensors:
        values = (
            torch.stack(tensors)
            .float()
            .cpu()
            .tolist()
        )

        numbers.update(zip(names, values))

    return numbers


def get_metrics():
    return {
        name: history[-1]
        for name, history in logs.items()
        if history
    }

def get_metrics_lists():
    return {
        name: history.copy()
        for name, history in logs.items()
        if history
    }

In [ ]:
## plot decoded images from list of seeds
@torch.no_grad()
def plot_constructions(seeds, patch_rows=None, patch_cols=None):

    constructs = decoder(seeds.to(device)).cpu()

    n = constructs.shape[0]

    fig, axes = plt.subplots(1, n, figsize=(n * 2, 4))

    if n == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        img = constructs[i].permute(1, 2, 0).float().clamp(0, 1)
        ax.imshow(img)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# plot objective losses
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


def moving_average(values, window=100):
    values = np.asarray(values)
    return np.convolve(
        values,
        np.ones(window) / window,
        mode="valid",
    )

def relative_change(values, eps=1e-12):
    values = np.asarray(
        values,
        dtype=float,
    )

    if values.size == 0:
        return values

    baseline = values[0]

    # Entire series is effectively zero
    if np.all(np.abs(values) < eps):
        return np.zeros_like(values)

    # Relative change is undefined if window starts at zero
    if abs(baseline) < eps:
        return np.full_like(
            values,
            np.nan,
        )

    return 100 * (
        values / baseline - 1
    )

def plot_objective_losses(
    history=None,
    window=1000,
    start=0,
    end=None,
    relative=False,
    show=True,
):
    if history is None:
        history = get_metrics_lists()
        
    fig, ax = plt.subplots(figsize=(10, 5))

    clean = moving_average(
        history["clean_term"],
        window,
    )[start:end]

    laplacian = moving_average(
        history["laplacian_term"],
        window,
    )[start:end]

    masked = moving_average(
        history["masked_term"],
        window,
    )[start:end]

    view = moving_average(
        history["view_term"],
        window,
    )[start:end]

    if relative:
        clean = relative_change(clean)
        laplacian = relative_change(laplacian)
        masked = relative_change(masked)
        view = relative_change(view)

    ax.plot(
        clean,
        label="clean recon",
    )

    ax.plot(
        laplacian,
        label="Laplacian",
    )

    # Uncomment for masked runs
    ax.plot(
        masked,
        label="masked prediction",
    )

    # Uncomment if you want view alignment here
    # ax.plot(
    #     view,
    #     label="view invariance",
    # )

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            lambda x, _: (
                f"start\n({start:,})"
                if abs(x) < 1
                else f"+{x/1000:.0f}k"
            )
        )
    )

    ax.set_xlabel("Training step")

    if relative:
        ax.set_ylabel("Change from window start (%)")
        ax.set_title(
            f"objective losses — relative change, "
            f"{window}-step moving average"
        )
        ax.axhline(
            0,
            linewidth=1,
            alpha=0.3,
        )
    else:
        ax.set_ylabel("Smoothed loss")
        ax.set_title(
            f"objective losses — "
            f"{window}-step moving average"
        )

    ax.legend()
    ax.grid(alpha=0.3)

    if show:
        plt.show()

    return fig

In [ ]:
def plot_regularization_losses(
    history=None,
    window=1000,
    start=0,
    end=None,
    relative=False,
    show=True,
):
    if history is None:
        history = get_metrics_lists()

    series = {
        "view alignment": "view_term",
        "SWD": "swd_term",
        "BWD": "bwd_term",
        "C_SWD": "c_swd_term",
        "T_SWD": "t_swd_term",
    }

    fig, ax = plt.subplots(figsize=(10, 5))

    for label, key in series.items():
        values = moving_average(
            history[key],
            window,
        )[start:end]

        if relative:
            values = relative_change(values)

        ax.plot(values, label=label)

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            lambda x, _: (
                f"start\n({start:,})"
                if abs(x) < 1
                else f"+{x/1000:.0f}k"
            )
        )
    )

    ax.set_xlabel("Training step")

    if relative:
        ax.set_ylabel("Change from window start (%)")
        ax.set_title(
            f"regularization losses — relative change, "
            f"{window}-step moving average"
        )
        ax.axhline(
            0,
            linewidth=1,
            alpha=0.3,
        )
    else:
        ax.set_ylabel("Smoothed loss")
        ax.set_title(
            f"regularization losses — "
            f"{window}-step moving average"
        )

    ax.legend()
    ax.grid(alpha=0.3)

    if show:
        plt.show()

    return fig

In [ ]:
def save_reconstruction_snapshot(masked=False):
    if (
        _active_run is not None
        and not _active_run.get(
            "disk_logging",
            True,
        )
    ):
        return False
    
    fig = render_reconstruction(
        masked=masked,
        show=False,
    )

    filename = (
        "latest_recon_masked.png"
        if masked
        else "latest_recon_clean.png"
    )

    fig.savefig(
        preview_dir / filename,
        dpi=130,
        bbox_inches="tight",
    )

    plt.close(fig)

    return True

In [ ]:
def save_loss_snapshots(
    window_full=1000,
    window_recent=500,
    recent_steps=20_000,
):

    if (
        _active_run is not None
        and not _active_run.get(
            "disk_logging",
            True,
        )
    ):
        return False
    history = get_metrics_lists()
    n_steps = len(history["clean_term"])

    fig = plot_objective_losses(
        window=window_full,
        start=0,
        show=False,
    )
    fig.savefig(
        preview_dir / "latest_objective_full.png",
        dpi=130,
        bbox_inches="tight",
    )
    plt.close(fig)

    fig = plot_regularization_losses(
        window=window_full,
        start=0,
        show=False,
    )
    fig.savefig(
        preview_dir / "latest_regularization_full.png",
        dpi=130,
        bbox_inches="tight",
    )
    plt.close(fig)

    recent_start = max(0, n_steps - recent_steps)

    fig = plot_objective_losses(
        window=window_recent,
        start=recent_start,
        relative=True,
        show=False,
    )
    fig.savefig(
        preview_dir / "latest_objective_recent.png",
        dpi=130,
        bbox_inches="tight",
    )
    plt.close(fig)

    fig = plot_regularization_losses(
        window=window_recent,
        start=recent_start,
        relative=True,
        show=False,
    )
    fig.savefig(
        preview_dir / "latest_regularization_recent.png",
        dpi=130,
        bbox_inches="tight",
    )
    plt.close(fig)

    return True

In [ ]:
#Helper: MULTIVIEW EVALUATION ##

@torch.no_grad()
def evaluate_multiview(max_batches=20, seed=1234):
    encoder_was_training = encoder.training
    decoder_was_training = decoder.training

    encoder.eval()
    decoder.eval()

    eval_device = next(encoder.parameters()).device
    cuda_devices = (
        [eval_device.index if eval_device.index is not None
         else torch.cuda.current_device()]
        if eval_device.type == "cuda" else []
    )

    def evaluate_split(loader):
        clean_sum = 0.0
        clean_count = 0
        masked_sum = 0.0
        masked_count = 0

        for batch_index, (images, _) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break

            images = images.to(device)
            B, channels, H, W = images.shape
            assert H % pr == 0 and W % pc == 0

            mask, keep_indices = make_mask(
                images,
                mask_ratio=encoder_patch_drop_ratio,
                return_keep_indices=True,
            )

            pixel_mask = (
                mask.reshape(B, 1, pr, pc)
                .repeat_interleave(H // pr, dim=2)
                .repeat_interleave(W // pc, dim=3)
            ).to(images.dtype)

            clean = decoder(encoder(images))
            masked = decoder(
                encoder(images, keep_indices=keep_indices)
            )

            clean_sum += F.mse_loss(
                clean, images, reduction="sum"
            ).item()
            clean_count += images.numel()

            masked_errors = F.mse_loss(
                masked, images, reduction="none"
            )
            masked_sum += (masked_errors * pixel_mask).sum().item()
            masked_count += pixel_mask.sum().item() * channels

        if clean_count == 0 or masked_count == 0:
            raise ValueError("Evaluation needs images and at least one masked pixel.")

        return clean_sum / clean_count, masked_sum / masked_count

    try:
        with torch.random.fork_rng(devices=cuda_devices):
            torch.random.default_generator.manual_seed(seed)
            if cuda_devices:
                with torch.cuda.device(cuda_devices[0]):
                    torch.cuda.manual_seed(seed)

            test_clean, test_missing = evaluate_split(test_dataloader)
            train_clean, train_missing = evaluate_split(train_dataloader)
    finally:
        encoder.train(encoder_was_training)
        decoder.train(decoder_was_training)

    return {
        "test clean": test_clean,
        "train clean": train_clean,
        "test missing": test_missing,
        "train missing": train_missing,
    }

In [ ]:
from collections import defaultdict
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import json
import os
import re
import torch


# ------------------------------------------------------------------
# Storage
# ------------------------------------------------------------------

PROJECT_ROOT = Path(os.environ["ML_CHECKPOINTS"]).parent
RUN_ROOT = PROJECT_ROOT / "training_runs" / "ae"
CHECKPOINT_ROOT = Path(os.environ["ML_CHECKPOINTS"]) / "ae"

RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


# Preserve state if this setup cell is rerun.
if "logs" not in globals():
    logs = defaultdict(list)

if "runs" not in globals():
    runs = {}

if "_active_run" not in globals():
    _active_run = None

if "current_run_name" not in globals():
    current_run_name = None

#regex for gating test/testing runs from polluting disk
import re

TEST_RUN_RE = re.compile(
    r"(^|[_-])(test|testing)([_-]|$)",
    re.IGNORECASE,
)

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def sanitize_run_name(name):
    name = re.sub(r"[^a-zA-Z0-9_.-]+", "_", name)
    return name.strip("_")


def get_metrics_lists():
    return {
        name: list(values)
        for name, values in logs.items()
        if len(values) > 0
    }


def history_steps(history):
    lengths = [
        len(values)
        for values in history.values()
        if hasattr(values, "__len__")
    ]

    return max(lengths, default=0)


def atomic_torch_save(data, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    torch.save(data, temporary)
    os.replace(temporary, destination)


def atomic_json_save(data, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            indent=2,
            default=str,
        )

    os.replace(temporary, destination)


def model_state_dict(model):
    # Avoid saving torch.compile's `_orig_mod.` prefix.
    state = model.state_dict()

    return {
        key.removeprefix("_orig_mod."): value
        for key, value in state.items()
    }


# ------------------------------------------------------------------
# Start
# ------------------------------------------------------------------

def start_run(
    name,
    config=None,
    disk_logging=None,
):
    global _active_run
    global current_run_name

    if _active_run is not None:
        raise RuntimeError(
            f"Run already active: {_active_run['run_id']}\n"
            "Call finish_run() before starting another."
        )

    orphan_steps = history_steps(
        get_metrics_lists()
    )

    if orphan_steps > 0:
        raise RuntimeError(
            f"Found {orphan_steps:,} live log steps "
            "with no active run.\n"
            "Do not clear them yet. "
            "Save or inspect them first."
        )

    timestamp = datetime.now().strftime(
        "%Y-%m-%d_%H%M%S_%f"
    )

    short_name = sanitize_run_name(name)
    run_id = f"{timestamp}_{short_name}"

    if disk_logging is None:
        disk_logging = (
            TEST_RUN_RE.search(short_name) is None
        )

    run_dir = (
        RUN_ROOT / run_id
        if disk_logging
        else None
    )

    if run_dir is not None:
        run_dir.mkdir(
            parents=True,
            exist_ok=False,
        )

    started_at = datetime.now().isoformat()
    run_config = deepcopy(config or {})

    metadata = {
        "run_id": run_id,
        "name": short_name,
        "status": "running",
        "started_at": started_at,
        "config": run_config,
    }

    _active_run = {
        "run_id": run_id,
        "name": short_name,
        "run_dir": run_dir,
        "disk_logging": disk_logging,
        "metadata": metadata,
        "started_at": started_at,
        "config": run_config,
        "last_saved_steps": 0,
    }

    current_run_name = run_id
    logs.clear()

    if disk_logging:
        atomic_json_save(
            metadata,
            run_dir / "metadata.json",
        )

    mode = (
        "disk logging enabled"
        if disk_logging
        else "memory only"
    )

    print(
        f"Started run:\n"
        f"{run_id}\n"
        f"{mode}"
    )

    return run_id

def finish_run():
    global _active_run
    global current_run_name

    history = deepcopy(
        get_metrics_lists()
    )

    steps = history_steps(history)
    finished_at = datetime.now().isoformat()

    # Preserve orphaned logs in notebook memory only.
    if _active_run is None:
        if steps == 0:
            print(
                "No active run or live logs."
            )
            return None

        timestamp = datetime.now().strftime(
            "%Y-%m-%d_%H%M%S_%f"
        )

        run_id = (
            f"{timestamp}_recovered_logs"
        )

        record = {
            "run_id": run_id,
            "name": "recovered_logs",
            "run_dir": None,
            "disk_logging": False,
            "status": "recovered",
            "started_at": None,
            "finished_at": finished_at,
            "final_steps": steps,
            "config": {},
            "previous_run_hint": (
                current_run_name
            ),
        }

    else:
        record = deepcopy(_active_run)
        run_id = record["run_id"]

        record["status"] = "finished"
        record["finished_at"] = finished_at
        record["final_steps"] = steps

        # Normal runs only update metadata.
        # Their history and checkpoint were already
        # saved by the training loop.
        if record.get(
            "disk_logging",
            True,
        ):
            run_dir = record.get("run_dir")

            if run_dir is not None:
                metadata = {
                    key: value
                    for key, value
                    in record.items()
                    if key not in {
                        "run_dir",
                        "metadata",
                    }
                }

                atomic_json_save(
                    metadata,
                    Path(run_dir)
                    / "metadata.json",
                )

    # Keep history available for notebook plots.
    runs[run_id] = {
        **record,
        "history": history,
    }

    logs.clear()
    _active_run = None
    current_run_name = None

    storage = (
        "disk checkpoint already saved"
        if record.get(
            "disk_logging",
            False,
        )
        else "kept in notebook memory only"
    )

    print(
        f"Finished run:\n"
        f"{run_id}\n"
        f"{steps:,} steps, {storage}."
    )

    return run_id

In [ ]:
#require promotable run
import re

MIN_PROMOTE_STEPS = 20_000

_TEST_RUN_RE = re.compile(
    r"(^|[_-])(test|testing)([_-]|$)",
    re.IGNORECASE,
)


def require_promotable_run():
    if _active_run is None:
        raise RuntimeError("No active run.")

    run_id = _active_run["run_id"]
    steps = history_steps(get_metrics_lists())

    if _TEST_RUN_RE.search(run_id):
        raise RuntimeError(
            f"Refusing to promote test run: {run_id}"
        )

    if steps < MIN_PROMOTE_STEPS:
        raise RuntimeError(
            f"Refusing to promote {run_id}: "
            f"{steps:,} < {MIN_PROMOTE_STEPS:,} steps."
        )

    return steps

In [ ]:
def resume_run(run_id):
    global _active_run
    global current_run_name
    global RUN_NAME

    if _active_run is not None:
        raise RuntimeError(
            f"Run already active: {_active_run['run_id']}"
        )

    run_dir = RUN_ROOT / run_id

    history_path = run_dir / "history.pt"
    checkpoint_path = run_dir / "checkpoint.pt"
    metadata_path = run_dir / "metadata.json"

    # Load files
    saved = torch.load(
        history_path,
        map_location="cpu",
        weights_only=False,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    with open(metadata_path) as f:
        metadata = json.load(f)

    # Restore model weights
    enc_result = encoder.load_state_dict(
        checkpoint["encoder"]
    )

    dec_result = decoder.load_state_dict(
        checkpoint["decoder"]
    )

    # Restore optimizer when available
    optimizer_restored = False

    if "optimizer" in checkpoint:
        ae_optimizer.load_state_dict(
            checkpoint["optimizer"]
        )
        optimizer_restored = True

    # Restore FULL history directly into live logs
    logs.clear()

    for name, values in saved["history"].items():
        logs[name].extend(values)

    # Restore run bookkeeping
    RUN_NAME = metadata["name"]
    current_run_name = run_id

    _active_run = {
        "run_id": run_id,
        "name": metadata["name"],
        "run_dir": run_dir,
        "started_at": metadata["started_at"],
        "config": deepcopy(
            metadata.get("config", {})
        ),
        "last_saved_steps": checkpoint["steps"],
        "last_epoch": checkpoint["epoch"],
        "resumed": True,
    }

    print("Resumed run:")
    print(f"  {run_id}")
    print(f"  epoch: {checkpoint['epoch']}")
    print(f"  steps: {checkpoint['steps']:,}")
    print(f"  encoder: {enc_result}")
    print(f"  decoder: {dec_result}")
    print(
        "  optimizer:",
        "restored" if optimizer_restored
        else "fresh state"
    )

    return checkpoint

In [ ]:
def merge_histories(base, new):
    keys = set(base) | set(new)

    merged = {}

    for key in keys:
        merged[key] = (
            list(base.get(key, []))
            + list(new.get(key, []))
        )

    return merged

In [ ]:
def start_z_gradient_diagnostics(z, **weighted_regularizers):
    if not TRACK_Z_DIAGNOSTICS:
        return None

    z.retain_grad()

    gradients = {}
    for name, loss in weighted_regularizers.items():
        gradients[name] = torch.autograd.grad(
            loss,
            z,
            retain_graph=True,
        )[0].detach()

    # retain_grad() can capture gradients from the diagnostic passes.
    # Clear them so the real backward supplies only the total gradient.
    z.grad = None

    return gradients


def finish_z_gradient_diagnostics(enc, diagnostic):
    if diagnostic is None:
        return {}

    if z.grad is None:
        raise RuntimeError("Call finish diagnostics after ae_loss.backward().")

    with torch.no_grad():
        g_remaining = z.grad.detach().float().clone()

        for gradient in diagnostic.values():
            g_remaining.sub_(gradient.float())

        remaining_rms = g_remaining.square().mean().sqrt()

        metrics = {"remaining_grad_rms": remaining_rms}
        for name, gradient in diagnostic.items():
            rms = gradient.float().square().mean().sqrt()
            metrics[f"{name}_grad_rms"] = rms
            metrics[f"{name}_grad_ratio"] = rms / remaining_rms.clamp_min(1e-12)

    return metrics

In [ ]:
def align_sigreg_grad(g_recon, g_sig, mode="plain", eps=1e-8):
    """
    Shape SIGReg's gradient relative to reconstruction.
    Projection is per image over the full latent [T, C].
    """
    shape = g_sig.shape

    gr = g_recon.flatten(1)
    gs = g_sig.flatten(1)

    # signed amount of SIGReg gradient parallel to recon gradient
    c = (
        (gs * gr).sum(dim=1, keepdim=True)
        / gr.square().sum(dim=1, keepdim=True).clamp_min(eps)
    )

    g_parallel = c * gr
    g_perp = gs - g_parallel

    if mode == "plain":
        out = gs

    elif mode == "sovereign":
        # SIGReg gets only directions locally neutral to recon
        out = g_perp

    elif mode == "protected":
        # remove only anti-recon parallel component
        out = gs - c.clamp(max=0) * gr

    elif mode == "reinforce":
        # keep only parallel component that helps recon
        out = c.clamp(min=0) * gr

    else:
        raise ValueError(f"unknown mode: {mode}")

    return out.view(shape)

In [ ]:
def save_run_snapshot(
    epoch=None,
    force=False,
):
    global _active_run

    if _active_run is None:
        raise RuntimeError(
            "No active run. "
            "Call start_run() or resume_run() first."
        )

    # Autosaves silently skip memory-only runs.
    if (
        not _active_run.get(
            "disk_logging",
            True,
        )
        and not force
    ):
        _active_run["last_epoch"] = epoch
        return False

    history = deepcopy(
        get_metrics_lists()
    )

    new_steps = history_steps(history)

    if new_steps == 0:
        raise RuntimeError(
            "Refusing to save an empty history."
        )

    run_dir = _active_run.get("run_dir")

    # force=True promotes a memory-only run
    # into a normal disk-backed run.
    if run_dir is None:
        run_dir = (
            RUN_ROOT
            / _active_run["run_id"]
        )

        run_dir.mkdir(
            parents=True,
            exist_ok=False,
        )

        metadata = deepcopy(
            _active_run["metadata"]
        )

        atomic_json_save(
            metadata,
            run_dir / "metadata.json",
        )

        _active_run["run_dir"] = run_dir
        _active_run["disk_logging"] = True

    history_path = (
        run_dir / "history.pt"
    )

    checkpoint_path = (
        run_dir / "checkpoint.pt"
    )

    if history_path.exists():
        existing = torch.load(
            history_path,
            map_location="cpu",
            weights_only=False,
        )

        old_steps = history_steps(
            existing.get(
                "history",
                {},
            )
        )

        if new_steps < old_steps:
            raise RuntimeError(
                f"Refusing to replace "
                f"{old_steps:,} saved steps "
                f"with {new_steps:,} steps."
            )

    atomic_torch_save(
        {
            "run_id": _active_run["run_id"],
            "epoch": epoch,
            "steps_per_epoch": len(
                train_dataloader
            ),
            "history": history,
        },
        history_path,
    )

    atomic_torch_save(
        {
            "run_id": _active_run["run_id"],
            "epoch": epoch,
            "steps": new_steps,
            "encoder": model_state_dict(
                encoder
            ),
            "decoder": model_state_dict(
                decoder
            ),
            "optimizer": (
                ae_optimizer.state_dict()
            ),
        },
        checkpoint_path,
    )

    _active_run["last_saved_steps"] = (
        new_steps
    )

    _active_run["last_epoch"] = epoch

    print(
        f"Saved epoch {epoch}: "
        f"{new_steps:,} steps "
        f"(model + optimizer)"
    )

    return True

In [ ]:
def capture_run_config():
    names = [
        "use_masked_view",
        "view_mode",
        "view_weight",
        "lr",
        "laplacian_weight",
        "swd_weight",
        "bwd_weight",
        "c_swd_weight",
        "t_swd_weight",
        "ae_emb_dim",
        "n_ae_latent_tokens",
        "emb_dim",
        "n_latent_tokens",
        "num_blocks",
        "n_heads",
        "head_size",
        "patch_size",
        "beta1",
        "beta2",
    ]

    config = {
        name: globals()[name]
        for name in names
        if name in globals()
    }

    config["batch_size"] = train_dataloader.batch_size

    return config

In [ ]:
#training readout
import time
import torch


class TrainingReadout:
    def __init__(
        self,
        step_metrics,
        report_every=500,
        watch="test clean",
        trend_epochs=5,
    ):
        # Mapping: display label -> key returned by your training step.
        self.step_metrics = step_metrics
        self.report_every = report_every
        self.watch = watch
        self.trend_epochs = trend_epochs
        self.total_steps = 0
        self.history = []

    @staticmethod
    def _number(value):
        if torch.is_tensor(value):
            return value.detach().item()
        return float(value)

    @staticmethod
    def _gain(old, new):
        if old == 0:
            return "—"
        return f"{100 * (old - new) / abs(old):+.1f}%"

    def start(self, title, total_epochs, evaluation):
        self.total_epochs = total_epochs
        self.history = [
            {key: self._number(value) for key, value in evaluation.items()}
        ]
        self.total_steps = 0

        print(f"\n{title} | {total_epochs} epochs")
        print(
            "START | "
            + " | ".join(
                f"{key} {value:.5f}"
                for key, value in self.history[0].items()
            )
        )

    def begin_epoch(self, epoch, phase=""):
        self.epoch = epoch
        self.epoch_steps = 0
        self.epoch_samples = 0
        self.epoch_sums = {}

        self.interval_steps = 0
        self.interval_samples = 0
        self.interval_sums = {}

        print(f"\nE{epoch}/{self.total_epochs} {phase}".rstrip())
        print(
            f"{'STEP':>8}"
            + "".join(f" {label:>10}" for label in self.step_metrics)
            + f" {'ms/step':>9} {'img/s':>8}"
        )

        self.epoch_start = time.perf_counter()
        self.interval_start = self.epoch_start

    def update(self, metrics, batch_size, last_batch=False):
        self.total_steps += 1
        self.epoch_steps += 1
        self.interval_steps += 1
        self.epoch_samples += batch_size
        self.interval_samples += batch_size

        for key in self.step_metrics.values():
            value = metrics[key]
            if torch.is_tensor(value):
                value = value.detach()

            # Keep tensor sums on-device; convert only when reporting.
            contribution = value * batch_size
            self.epoch_sums[key] = self.epoch_sums.get(key, 0) + contribution
            self.interval_sums[key] = (
                self.interval_sums.get(key, 0) + contribution
            )

        if self.interval_steps >= self.report_every or last_batch:
            averages = {
                key: self._number(value / self.interval_samples)
                for key, value in self.interval_sums.items()
            }

            # Read scalar values before measuring elapsed time so GPU work
            # needed for these metrics has finished.
            now = time.perf_counter()
            elapsed = max(now - self.interval_start, 1e-9)
            self.train_seconds = now - self.epoch_start

            print(
                f"{self.total_steps:8d}"
                + "".join(
                    f" {averages[key]:10.5f}"
                    for key in self.step_metrics.values()
                )
                + f" {1000 * elapsed / self.interval_steps:9.0f}"
                + f" {self.interval_samples / elapsed:8.0f}"
            )

            self.interval_steps = 0
            self.interval_samples = 0
            self.interval_sums = {}
            self.interval_start = time.perf_counter()

    def end_epoch(self, evaluation):
        current = {
            key: self._number(value) for key, value in evaluation.items()
        }
        previous = self.history[-1]
        self.history.append(current)

        n = self.trend_epochs
        older = self.history[-n - 1] if len(self.history) > n else None

        best_index = min(
            range(len(self.history)),
            key=lambda i: self.history[i][self.watch],
        )
        best = self.history[best_index][self.watch]
        best_at = "start" if best_index == 0 else f"E{best_index}"

        value = current[self.watch]
        gain_1 = self._gain(previous[self.watch], value)
        gain_n = self._gain(older[self.watch], value) if older else "—"

        print(
            f"\nE{self.epoch}/{self.total_epochs}"
            f" | {self.watch} {value:.5f}"
            f" | gain {gain_1}/1ep {gain_n}/{n}ep"
            f" | best {best:.5f} @{best_at}"
            f" | {self.train_seconds / 60:.2f} min"
        )
        print(
            "Other eval | "
            + " | ".join(
                f"{key} {value:.5f}"
                for key, value in current.items()
                if key != self.watch
            )
        )

In [ ]:
# --------------------Def Init Models (and optimizers) --------------------


def init_models():
    
    encoder = Encoder(
        patch_size,
        c,
        ae_emb_dim,
        emb_dim,
        n_ae_latent_tokens,
        n_latent_tokens,
        num_blocks,
        n_heads,
        head_size,
    )
    
    decoder = Decoder(
        patch_size,
        c,
        ae_emb_dim,
        emb_dim,
        n_ae_latent_tokens,
        n_latent_tokens,
        num_blocks,
        n_heads,
        head_size,
    )
    
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    encoder.to(device)
    decoder.to(device)

    # encoder.sparse_middle_transformer.sparse_middle = use_sparse_middle
    # decoder.sparse_middle_transformer.sparse_middle = use_sparse_middle
    
    # -------------------- Optimizers --------------------
    
    ae_parameters = (
        list(encoder.parameters())
        + list(decoder.parameters())
    )
    n_ae_parameters  = sum(p.numel() for p in ae_parameters)
    print(f"{n_ae_parameters:,} ae parameters")
    
    ae_optimizer = torch.optim.AdamW(
        ae_parameters,
        lr=lr,
        betas=(beta1, beta2),
        fused=True,
    )

    return (
        encoder,
        decoder,
        ae_optimizer,
        device,
    )


In [ ]:
finish_run() # copy history to runs for plotting

In [ ]:
######################################################################################################################################
############################################################ init models #############################################################
######################################################################################################################################


SEED = 44

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
train_generator.manual_seed(DATA_SEED)
torch._dynamo.reset()

(
    encoder,
    decoder,
    ae_optimizer,
    device,
) = init_models()

In [ ]:
# resume_run(
#     "2026-09-18_170535_353997_ae_14M_tok_mask_deep"
# )

In [ ]:
start_run(
    RUN_NAME,
    config=capture_run_config(),
)

In [ ]:
render_reconstruction(); #test model recon

In [ ]:
################## Training Step Methods ########################

import copy
import torch
import torch.nn.functional as F

def make_color_mask(images, drop_one_prob=0.1):
    """Per image: keep RGB, or zero one random channel."""
    B = images.shape[0]

    channel = torch.randint(3, (B,), device=images.device)

    drop_channel = F.one_hot(
        channel,
        num_classes=3,
    ).bool()[:, :, None, None]

    use_drop = (
        torch.rand(B, 1, 1, 1, device=images.device)
        < drop_one_prob
    )

    return (~(use_drop & drop_channel)).to(images.dtype)

def make_mask(
    images,
    mask_ratio=encoder_patch_drop_ratio,
    return_keep_indices=False,
):
    B = images.shape[0]
    T = pr * pc
    n_mask = int(mask_ratio * T)

    mask_indices = torch.rand(B, T, device=images.device).topk(
        k=n_mask, dim=1, largest=False,
    ).indices
    mask = torch.zeros(B, T, dtype=torch.bool, device=images.device)
    mask.scatter_(dim=1, index=mask_indices, value=True)

    if return_keep_indices:
        n_keep = T - n_mask
        keep_indices = mask.int().argsort(
            dim=1, stable=True,
        )[:, :n_keep]
        return mask, keep_indices

    return mask


def make_pixel_mask(mask, images):
    B, _, H, W = images.shape
    return (
        mask.reshape(B, 1, pr, pc)
        .repeat_interleave(H // pr, dim=2)
        .repeat_interleave(W // pc, dim=3)
        .to(images.dtype)
    )


def make_crop_grid(images, crop_min=0.2, crop_max=0.9, edge_snap=edge_snap):
    B, _, H, W = images.shape
    device = images.device

    ratio = torch.empty(B, 1, 1, device=device).uniform_(
        crop_min, crop_max
    )
    size = (ratio * min(H, W)).floor().clamp(min=1)

    half = (size / 2).floor()

    center_y = (
        torch.rand(B, 1, 1, device=device) * (H + size)
    ).floor() - half
    
    center_x = (
        torch.rand(B, 1, 1, device=device) * (W + size)
    ).floor() - half
   
    top = torch.minimum((center_y - half).clamp(min=0), H - size)
    left = torch.minimum((center_x - half).clamp(min=0), W - size)

    # small crops snapped to the image border, per axis, to offset large crops' centre bias
    small = size <= 0.5 * min(H, W)
    snap_y = small & (torch.rand_like(size) < edge_snap)
    snap_x = small & (torch.rand_like(size) < edge_snap)
    top = torch.where(snap_y, torch.where(torch.rand_like(size) < 0.5, 0.0, H - size), top)
    left = torch.where(snap_x, torch.where(torch.rand_like(size) < 0.5, 0.0, W - size), left)
    

    ys = torch.linspace(0, 1, H, device=device)
    xs = torch.linspace(0, 1, W, device=device)
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")

    margin = 0.3  # Source pixels per side; previous test used 0.5.
    gy = top - margin + gy.unsqueeze(0) * (size - 1 + 2 * margin)
    gx = left - margin + gx.unsqueeze(0) * (size - 1 + 2 * margin)

    gy = gy / max(H - 1, 1) * 2 - 1
    gx = gx / max(W - 1, 1) * 2 - 1

    return torch.stack((gx, gy), dim=-1).float()


def crop_scale(images, grid):
    # FP32 coordinates avoid BF16 rounding of crop positions.
    with torch.autocast(device_type="cuda", enabled=False):
        return F.grid_sample(
            images.float(), grid, mode="bilinear",
            padding_mode="border", align_corners=True,
        )


def make_train_step(crop_min=0.2, crop_max=0.9):
    for name, ratio in {
        "crop_min": crop_min,
        "crop_max": crop_max,
    }.items():
        if not 0.0 < ratio <= 1.0:
            raise ValueError(
                f"{name} must be in (0, 1]"
            )
    # Deep-copy the underlying module if the encoder is torch.compile-wrapped.
    source_encoder = getattr(encoder, "_orig_mod", encoder)
    Ef = copy.deepcopy(source_encoder)
    Ef.requires_grad_(False)
    Ef.eval()
    for parameter in Ef.parameters():
        parameter.grad = None

    # Keep tensor identities stable, including when compiling train_step.
    state_pairs = tuple(zip(Ef.parameters(), source_encoder.parameters()))
    state_pairs += tuple(zip(Ef.buffers(), source_encoder.buffers()))

    def train_step(images, optimizer, view_weight):
        optimizer.zero_grad(set_to_none=True)

        # Snapshot E before any forward or optimizer update this step.
        with torch.no_grad():
            for frozen_tensor, live_tensor in state_pairs:
                frozen_tensor.copy_(live_tensor)

        _, keep_indices = make_mask(
            images,
            mask_ratio=encoder_patch_drop_ratio,
            return_keep_indices=True,
        )

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            z_clean = encoder(images)
        
            B = z_clean.shape[0]
            t = torch.rand(B, 1, 1, device=z_clean.device)
            t = clean_path_latent_noise_min + t * (
                clean_path_latent_noise_max - clean_path_latent_noise_min
            )
            t *= (
                torch.rand(B, 1, 1, device=z_clean.device)
                < clean_path_latent_noise_prob
            )
        
            z_clean_perturbed = torch.lerp(
                z_clean,
                torch.randn_like(z_clean),
                t.to(z_clean.dtype),
            )
            recon_clean = decoder(z_clean_perturbed)
        
            z_masked = encoder(
                images,
                keep_indices=keep_indices,
            )
            recon_masked = decoder(z_masked)
        
            # One shared crop.
            crop_grid = make_crop_grid(
                images,
                crop_min=crop_min,
                crop_max=crop_max,
            )
            

            all_images = torch.cat(
                [images, recon_clean, recon_masked],
                dim=0,
            )  # (3B, C, H, W)
            
            all_grids = crop_grid.repeat(3, 1, 1, 1)
            
            all_crops = crop_scale(
                all_images,
                all_grids,
            )
            
            color_mask = make_color_mask(
                images,
                color_drop_one_prob,
            )  # (B, C, 1, 1)
            
            all_color_masks = color_mask.repeat(3, 1, 1, 1)
            
            all_crops = all_crops * all_color_masks
            
            crop_real, crop_clean, crop_masked = all_crops.chunk(3, dim=0)

            with torch.no_grad():
                crop_target = Ef(crop_real)
            
            judged = Ef(torch.cat(
                [crop_clean, crop_masked],
                dim=0,
            ))
            
            clean_judged, masked_judged = judged.chunk(2, dim=0)



        z_clean = z_clean.float()
        recon_clean = recon_clean.float()
        consistency_loss = F.mse_loss(
            clean_judged.detach().float(), masked_judged.float(),
        )
        clean_crop_loss = F.mse_loss(
            clean_judged.float(), crop_target.float(),
        )
        masked_crop_loss = F.mse_loss(
            masked_judged.float(), crop_target.float(),
        )

        view_term = view_weight * (
            consistency_loss + clean_crop_loss + masked_crop_loss
        ) / 3.0


        # Monitoring only: raw losses under the existing chart keys.
        with torch.no_grad():
            clean_term = F.mse_loss(recon_clean, images)
            masked_term = F.mse_loss(recon_masked.float(), images)
        
            laplacian_term = F.mse_loss(
                laplacian(recon_clean), laplacian(images),
            )

        # Same regularizers, prior, and weights; only unperturbed clean z.
        prior = torch.randn_like(z_clean)
        clean_swd = sliced_wasserstein_distance(z_clean, prior, bulk_sigma=None)
        clean_bwd = basis_wasserstein_distance(z_clean, prior, bulk_sigma=None)
        clean_c_swd = channel_sliced_wasserstein_distance(z_clean, prior, bulk_sigma=None)
        clean_t_swd = token_sliced_wasserstein_distance(z_clean, prior, bulk_sigma=None)

        swd_term = swd_weight * clean_swd
        bwd_term = bwd_weight * clean_bwd
        c_swd_term = c_swd_weight * clean_c_swd
        t_swd_term = t_swd_weight * clean_t_swd

        ae_loss = (
            view_term + swd_term + bwd_term + c_swd_term + t_swd_term
        )
        ae_loss.backward()
        optimizer.step()

        zero = z_clean.new_zeros(())
        return {
            "ae_loss": ae_loss.detach(),
            "clean_term": clean_term.detach(),
            "masked_term": masked_term.detach(),
            "laplacian_term": laplacian_term.detach(),
            "view_term": view_term.detach(),
            "swd_term": swd_term.detach(),
            "bwd_term": bwd_term.detach(),
            "c_swd_term": c_swd_term.detach(),
            "t_swd_term": t_swd_term.detach(),
            "clean_swd": clean_swd.detach(),
            "masked_swd": zero,
            "clean_bwd": clean_bwd.detach(),
            "masked_bwd": zero,
            "clean_c_swd": clean_c_swd.detach(),
            "masked_c_swd": zero,
            "clean_t_swd": clean_t_swd.detach(),
            "masked_t_swd": zero,
        }

    return train_step



In [ ]:
#make and compile the training steps
train_step = torch.compile(make_train_step(
    crop_min=crop_min, 
    crop_max=crop_max,
))

In [ ]:
# fixed_micro_batch_size = 64
# fixed_micro_batch = next(iter(train_dataloader))[0][:fixed_micro_batch_size].to(device)
# fixed_mask = make_mask(fixed_micro_batch)

In [ ]:
# ############## overfit micro batch ####################
# num_epochs = 5
# report_every = 500
# eval_batches = 20  # None = full loaders

# encoder.requires_grad_(True)
# decoder.requires_grad_(True)

# #SET TO FALSE FOR DENSE MIDDLE
# train_sparse_middle = False
# # encoder.sparse_middle_transformer.sparse_middle = train_sparse_middle
# # decoder.sparse_middle_transformer.sparse_middle = train_sparse_middle

# train_optimizer = ae_optimizer


# readout = TrainingReadout(
#     step_metrics={
#         "clean": "clean_term",
#         "missing": "masked_term",
#         "view": "view_term",
#         "swd": "swd_term",
#         "bwd": "bwd_term",
#         "c_swd": "c_swd_term",
#         "t_swd": "t_swd_term",
#     },
#     report_every=report_every,
#     watch="test clean",
#     trend_epochs=5,
# )

# readout.start(
#     title=(
#         f"{current_run_name} | "
#         f"masked={use_masked_view} | "
#         f"view={view_mode} | "
#         f"view_weight={view_weight:g}"
#     ),
#     total_epochs=num_epochs,
#     evaluation=evaluate_multiview(
#         max_batches=eval_batches,
#     ),
# )

# steps_per_epoch = len(train_dataloader)

# for epoch in range(num_epochs):
#     encoder.train()
#     decoder.train()

#     readout.begin_epoch(
#         epoch + 1,
#         phase="TRAIN",
#     )

#     for batch_step, (images, _) in enumerate(train_dataloader):

#         images = images.to(
#             device,
#             non_blocking=True,
#         )

#         metrics = train_step(
#             fixed_micro_batch,
#             optimizer=train_optimizer,
#             view_weight=view_weight,
#         )

#         metric_values = collect_metrics(metrics)

#         log_metrics(**metric_values)

#         readout.update(
#             metrics,
#             batch_size=images.shape[0],
#             last_batch=(batch_step + 1 == steps_per_epoch),
#         )

#     #cell display
#     readout.end_epoch(
#         evaluate_multiview(
#             max_batches=eval_batches,
#         )
#     )

#     #saving/logging
#     save_loss_snapshots()
#     save_reconstruction_snapshot(masked=False)
#     save_reconstruction_snapshot(masked=True)
    
#     completed_epoch = (
#         _active_run.get("last_epoch", 0) + 1
#     )
    
#     save_run_snapshot(
#         epoch=completed_epoch,
#     )
    

#     plt.close("all")

In [ ]:
# for image in fixed_micro_batch:
#     reconstruct_images(image[None, ...])

In [ ]:
from pathlib import Path

preview_dir = Path("training_runs") / current_run_name / "latest"
preview_dir.mkdir(parents=True, exist_ok=True)

print(preview_dir)

In [ ]:
# new_lr = 1e-5

# for i, group in enumerate(train_optimizer.param_groups):
#     old_lr = group["lr"]
#     group["lr"] = new_lr
#     print(f"Group {i}: {old_lr:g} → {new_lr:g}")

In [ ]:
## AE EXPERIMENT TRAINING LOOP ##

num_epochs = 200
report_every = 500
eval_batches = 20  # None = full loaders

encoder.requires_grad_(True)
decoder.requires_grad_(True)

#SET TO FALSE FOR DENSE MIDDLE
train_sparse_middle = False
# encoder.sparse_middle_transformer.sparse_middle = train_sparse_middle
# decoder.sparse_middle_transformer.sparse_middle = train_sparse_middle

train_optimizer = ae_optimizer


readout = TrainingReadout(
    step_metrics={
        "clean": "clean_term",
        "laplacian": "laplacian_term",
        "view": "view_term",
        "swd": "swd_term",
        "bwd": "bwd_term",
        "c_swd": "c_swd_term",
        "t_swd": "t_swd_term",
    },
    report_every=report_every,
    watch="test clean",
    trend_epochs=5,
)

readout.start(
    title=(
        f"{current_run_name} | "
        f"masked={use_masked_view} | "
        f"view_weight={view_weight:g}"
    ),
    total_epochs=num_epochs,
    evaluation=evaluate_multiview(
        max_batches=eval_batches,
    ),
)

steps_per_epoch = len(train_dataloader)

for epoch in range(num_epochs):
    encoder.train()
    decoder.train()

    readout.begin_epoch(
        epoch + 1,
        phase="TRAIN",
    )

    for batch_step, (images, _) in enumerate(train_dataloader):

        images = images.to(
            device,
            non_blocking=True,
        )

        metrics = train_step(
            images,
            optimizer=train_optimizer,
            view_weight=view_weight,
        )

        metric_values = collect_metrics(metrics)

        log_metrics(**metric_values)

        readout.update(
            metrics,
            batch_size=images.shape[0],
            last_batch=(batch_step + 1 == steps_per_epoch),
        )

    #cell display
    readout.end_epoch(
        evaluate_multiview(
            max_batches=eval_batches,
        )
    )

    #saving/logging
    save_loss_snapshots()
    save_reconstruction_snapshot(masked=False)
    save_reconstruction_snapshot(masked=True)
    
    completed_epoch = (
        _active_run.get("last_epoch", 0) + 1
    )
    
    save_run_snapshot(
        epoch=completed_epoch,
    )

    

    plt.close("all")

In [ ]:
# Clean test reconstruction: full STL10 test set, eval mode (no RoPE jitter), no mask,
# no latent noise, no crop, no colour drop. Straight image -> z -> image.
@torch.no_grad()
def eval_clean_test(use_bf16=True, max_batches=None):
    enc_was, dec_was = encoder.training, decoder.training
    encoder.eval()
    decoder.eval()
    se, n, n_imgs = 0.0, 0, 0
    try:
        for i, (images, _) in enumerate(test_dataloader):
            if max_batches is not None and i >= max_batches:
                break
            images = images.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                recon = decoder(encoder(images))
            se += F.mse_loss(recon.float(), images.float(), reduction="sum").item()
            n += images.numel()
            n_imgs += images.shape[0]
    finally:
        encoder.train(enc_was)
        decoder.train(dec_was)
    mse = se / n
    psnr = 10 * math.log10(1.0 / mse)
    print(f"clean test MSE {mse:.5f}  PSNR {psnr:.2f} dB  ({n_imgs:,} images, {'bf16' if use_bf16 else 'fp32'})")
    return mse


import math
clean_test_mse = eval_clean_test()

In [ ]:
history = get_metrics_lists()


current_steps = len(history['clean_term'])
steps_per_epoch = len(train_dataloader)

epochs = current_steps / steps_per_epoch

print(f"steps:           {current_steps:,}")
print(f"steps / epoch:   {steps_per_epoch:,}")
print(f"epochs trained:  {epochs:.2f}")

In [ ]:
# List available saved histories

from pathlib import Path
import torch
from datetime import datetime

HISTORY_ROOT = Path("<home>/projects/Jupyter_AI/training_runs/ae")

history_files = sorted(
    HISTORY_ROOT.rglob("history.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

print(f"{len(history_files)} histories found:\n")

for i, p in enumerate(history_files):
    try:
        d = torch.load(p, map_location="cpu", weights_only=False)

        run_id = d.get("run_id", p.parent.name)
        epoch = d.get("epoch", "?")
        steps = len(d.get("history", {}).get("clean_term", []))

        print(
            f"[{i:2d}] "
            f"epoch {str(epoch):>4} | "
            f"steps {steps:>8,} | "
            f"{run_id}"
        )

    except Exception as e:
        print(f"[{i:2d}] ERROR | {p.parent.name} | {e}")

In [ ]:
# Compare histories helper
import numpy as np
import torch
import matplotlib.pyplot as plt


def load_saved_history(index):
    """Load one history from the indexed history_files list."""
    p = history_files[index]

    d = torch.load(
        p,
        map_location="cpu",
        weights_only=False,
    )

    return {
        "label": d.get("run_id", p.parent.name),
        "history": d["history"],
        "steps_per_epoch": d.get("steps_per_epoch", None),
        "epoch": d.get("epoch", None),
        "path": p,
    }


def _moving_average(x, window):
    x = np.asarray(x, dtype=np.float64)

    if window <= 1:
        return x

    if len(x) < window:
        return np.array([])

    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="valid")


def compare_histories(
    current_history,
    others=(),
    metric="clean_term",
    current_label="current",
    window=1_000,
    start=0,
    end=None,
    x_axis="steps",        # "steps" or "epochs"
    current_steps_per_epoch=None,
    relative=False,
    figsize=(11, 6),
):
    """
    Compare current history against saved histories.

    others:
        iterable of results from load_saved_history()

    relative:
        False -> raw smoothed loss
        True  -> each curve starts at 1.0
    """

    runs = [{
        "label": current_label,
        "history": current_history,
        "steps_per_epoch": current_steps_per_epoch,
    }]

    runs.extend(others)

    fig, ax = plt.subplots(figsize=figsize)

    for run in runs:

        h = run["history"]

        if metric not in h:
            print(f"Skipping {run['label']}: no '{metric}'")
            continue

        y = _moving_average(
            h[metric],
            window,
        )

        y = y[start:end]

        if len(y) == 0:
            print(f"Skipping {run['label']}: no points after slicing")
            continue

        if relative:
            y = y / y[0]

        # x positions correspond to original training steps
        x = np.arange(len(y)) + start + window - 1

        if x_axis == "epochs":
            spe = run.get("steps_per_epoch")

            if spe is None:
                raise ValueError(
                    f"{run['label']} has no steps_per_epoch"
                )

            x = x / spe

        ax.plot(
            x,
            y,
            label=run["label"],
            linewidth=1.8,
        )

    ax.set_title(metric)

    ax.set_xlabel(
        "Epoch"
        if x_axis == "epochs"
        else "Training step"
    )

    ax.set_ylabel(
        "Relative loss"
        if relative
        else metric
    )

    ax.grid(alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Pick histories by index from the listing above

# h1 = load_saved_history(2)
# h2 = load_saved_history(8)
# h3 = load_saved_history(16)
# h4 = load_saved_history(23)


In [ ]:
# compare_histories(
#     current_history=history,
#     current_label="current",
#     current_steps_per_epoch=len(train_dataloader),

#     others=[
#         h1,
#         h2,
#         h3,
#         h4,
#     ],

#     metric="clean_term",
#     window=1_00,
#     start=0_000,
#     end=int(current_steps*1.25),
#     x_axis="steps",
#     relative=False,
# )

In [ ]:
# compare_histories(
#     current_history=history,
#     current_label="current",
#     current_steps_per_epoch=len(train_dataloader),

#     others=[
#         h1,
#         h2,
#         h3,
#         h4,
#     ],

#     metric="clean_term",
#     window=1_000,
#     start=max(0, current_steps-30_000),
#     end=current_steps+10_000,
#     x_axis="steps",
#     relative=True,
# )

In [ ]:
plot_objective_losses(
    history,
    window=1_00,
    start=0,
    relative=False,
);

In [ ]:
plot_objective_losses(
    history,
    window=10000,
    start=75_000, #max(000, current_steps-50_000),
    relative=True,
);

In [ ]:
plot_regularization_losses(
    history,
    window=1000,
    start=0,
    relative=False,
);

In [ ]:
plot_regularization_losses(
    history,
    window=1_000,
    start=75000,#max(0, current_steps-20_000),
    relative=True,
);

In [ ]:
# latent stats: plot_z_dashboard
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def plot_z_dashboard(
    z,
    n_ref=256,
    sort_by="individual",   # "mean", "std", "individual"
    sort_abs=True,          # True = magnitude, False = signed
    heatmap_percentile=99,  # robust data-dependent color scaling
):
    """
    z: (B, L, D)

    sort_by:
        "mean"       -> both heatmaps use mean-based row/column ordering
        "std"        -> both heatmaps use std-based row/column ordering
        "individual" -> mean heatmap sorts by mean, std heatmap sorts by std

    sort_abs:
        Mean:
            True  -> sort by |mean|, farthest from 0 first
            False -> sort by signed mean, positive -> negative

        Std:
            True  -> sort by |std - 1|, farthest from 1 first
            False -> sort by std - 1, high std -> low std

    heatmap_percentile:
        Robust percentile used for data-dependent heatmap limits.
        Mean stays symmetric around 0.
        Std uses its own low/high percentiles.
    """

    assert sort_by in ("mean", "std", "individual")
    assert 50 < heatmap_percentile <= 100

    z = z.detach().float().cpu()
    assert z.ndim == 3, f"Expected (B, L, D), got {z.shape}"

    B, L, D = z.shape
    assert B >= 2, "Need at least B=2"

    # ------------------------------------------------------------
    # REAL STATS
    # ------------------------------------------------------------

    cell_mean = z.mean(dim=0)               # (L, D)
    cell_std = z.std(dim=0, unbiased=True)  # (L, D)

    # What gets displayed in the side/top charts
    if sort_abs:
        token_mean = cell_mean.abs().mean(dim=1)
        emb_mean = cell_mean.abs().mean(dim=0)

        token_std = (cell_std - 1).abs().mean(dim=1)
        emb_std = (cell_std - 1).abs().mean(dim=0)

    else:
        token_mean = cell_mean.mean(dim=1)
        emb_mean = cell_mean.mean(dim=0)

        token_std = (cell_std - 1).mean(dim=1)
        emb_std = (cell_std - 1).mean(dim=0)

    # ------------------------------------------------------------
    # SORT ORDERS
    # ------------------------------------------------------------

    mean_row_order = token_mean.argsort(descending=True)
    mean_col_order = emb_mean.argsort(descending=True)

    std_row_order = token_std.argsort(descending=True)
    std_col_order = emb_std.argsort(descending=True)

    if sort_by == "mean":
        mean_rows = mean_row_order
        mean_cols = mean_col_order

        std_rows = mean_row_order
        std_cols = mean_col_order

    elif sort_by == "std":
        mean_rows = std_row_order
        mean_cols = std_col_order

        std_rows = std_row_order
        std_cols = std_col_order

    else:  # individual
        mean_rows = mean_row_order
        mean_cols = mean_col_order

        std_rows = std_row_order
        std_cols = std_col_order

    # ------------------------------------------------------------
    # RANDN REFERENCE
    # ------------------------------------------------------------

    ref_token_mean = []
    ref_emb_mean = []
    ref_token_std = []
    ref_emb_std = []

    for _ in range(n_ref):
        r = torch.randn(B, L, D)

        r_mean = r.mean(dim=0)
        r_std = r.std(dim=0, unbiased=True)

        if sort_abs:
            ref_token_mean.append(r_mean.abs().mean(dim=1))
            ref_emb_mean.append(r_mean.abs().mean(dim=0))

            ref_token_std.append((r_std - 1).abs().mean(dim=1))
            ref_emb_std.append((r_std - 1).abs().mean(dim=0))

        else:
            ref_token_mean.append(r_mean.mean(dim=1))
            ref_emb_mean.append(r_mean.mean(dim=0))

            ref_token_std.append((r_std - 1).mean(dim=1))
            ref_emb_std.append((r_std - 1).mean(dim=0))

    ref_token_mean = torch.stack(ref_token_mean)
    ref_emb_mean = torch.stack(ref_emb_mean)
    ref_token_std = torch.stack(ref_token_std)
    ref_emb_std = torch.stack(ref_emb_std)

    def summarize_ref(x):
        return (
            x.mean(dim=0),
            x.quantile(0.05, dim=0),
            x.quantile(0.95, dim=0),
        )

    token_mean_ref, token_mean_lo, token_mean_hi = summarize_ref(ref_token_mean)
    emb_mean_ref, emb_mean_lo, emb_mean_hi = summarize_ref(ref_emb_mean)

    token_std_ref, token_std_lo, token_std_hi = summarize_ref(ref_token_std)
    emb_std_ref, emb_std_lo, emb_std_hi = summarize_ref(ref_emb_std)

    # ------------------------------------------------------------
    # DATA-DEPENDENT HEATMAP LIMITS
    # ------------------------------------------------------------

    q = heatmap_percentile / 100.0
    q_lo = 1.0 - q

    # Mean:
    # keep zero as the visual center, but ignore extreme outliers
    mean_hm_lim = torch.quantile(cell_mean.abs().flatten(), q).item()

    if mean_hm_lim <= 1e-12:
        mean_hm_lim = 1e-12

    mean_hm_lo = -mean_hm_lim
    mean_hm_hi = mean_hm_lim

    # Std:
    # let the actual data determine both ends of the scale.
    # This makes a collapsed 0.000-0.023 latent visibly structured,
    # while a healthy ~0.8-1.2 latent gets its own useful range.
    std_flat = cell_std.flatten()

    std_hm_lo = torch.quantile(std_flat, q_lo).item()
    std_hm_hi = torch.quantile(std_flat, q).item()

    # Std cannot be negative
    std_hm_lo = max(0.0, std_hm_lo)

    # Guard against a perfectly / nearly constant tensor
    if std_hm_hi - std_hm_lo <= 1e-12:
        pad = max(abs(std_hm_hi) * 0.01, 1e-6)
        std_hm_lo = max(0.0, std_hm_lo - pad)
        std_hm_hi = std_hm_hi + pad

    # ------------------------------------------------------------
    # FIGURE
    # ------------------------------------------------------------

    fig = plt.figure(figsize=(18, 8), constrained_layout=True)

    gs = fig.add_gridspec(
        2, 4,
        height_ratios=[1.15, 4.0],
        width_ratios=[1.2, 4.0, 1.2, 4.0],
        wspace=0.05,
        hspace=0.05,
    )

    ax_blank1 = fig.add_subplot(gs[0, 0])
    ax_emb_mean = fig.add_subplot(gs[0, 1])

    ax_blank2 = fig.add_subplot(gs[0, 2])
    ax_emb_std = fig.add_subplot(gs[0, 3])

    ax_token_mean = fig.add_subplot(gs[1, 0])
    ax_mean_hm = fig.add_subplot(gs[1, 1])

    ax_token_std = fig.add_subplot(gs[1, 2])
    ax_std_hm = fig.add_subplot(gs[1, 3])

    ax_blank1.axis("off")
    ax_blank2.axis("off")

    # ------------------------------------------------------------
    # HELPER PLOTS
    # ------------------------------------------------------------

    def plot_top_summary(ax, values, ref, lo, hi, title):
        x = torch.arange(len(values)).numpy()

        ax.bar(x, values.numpy(), color="grey", width=1.0)
        ax.plot(x, ref.numpy(), color="green", linewidth=1.5)

        ax.fill_between(
            x,
            lo.numpy(),
            hi.numpy(),
            color="yellow",
            alpha=0.25,
        )

        ax.set_title(title, fontsize=10)
        ax.set_xlim(-0.5, len(values) - 0.5)

        ax.tick_params(axis="x", labelbottom=False)
        ax.tick_params(axis="y", labelsize=8)

    def plot_side_summary(ax, values, ref, lo, hi, title):
        y = torch.arange(len(values)).numpy()

        ax.barh(y, values.numpy(), color="grey", height=1.0)
        ax.plot(ref.numpy(), y, color="green", linewidth=1.5)

        ax.fill_betweenx(
            y,
            lo.numpy(),
            hi.numpy(),
            color="yellow",
            alpha=0.25,
        )

        ax.set_title(title, fontsize=10)
        ax.set_ylim(len(values) - 0.5, -0.5)

        ax.tick_params(axis="y", labelleft=False)
        ax.tick_params(axis="x", labelsize=8)

    # ------------------------------------------------------------
    # MINI CHARTS
    # ------------------------------------------------------------

    plot_top_summary(
        ax_emb_mean,
        emb_mean[mean_cols],
        emb_mean_ref[mean_cols],
        emb_mean_lo[mean_cols],
        emb_mean_hi[mean_cols],
        "emb-mean",
    )

    plot_side_summary(
        ax_token_mean,
        token_mean[mean_rows],
        token_mean_ref[mean_rows],
        token_mean_lo[mean_rows],
        token_mean_hi[mean_rows],
        "token-mean",
    )

    plot_top_summary(
        ax_emb_std,
        emb_std[std_cols],
        emb_std_ref[std_cols],
        emb_std_lo[std_cols],
        emb_std_hi[std_cols],
        "emb-std",
    )

    plot_side_summary(
        ax_token_std,
        token_std[std_rows],
        token_std_ref[std_rows],
        token_std_lo[std_rows],
        token_std_hi[std_rows],
        "token-std",
    )

    # ------------------------------------------------------------
    # HEATMAPS
    # ------------------------------------------------------------

    mean_heatmap_sorted = cell_mean[mean_rows][:, mean_cols]
    std_heatmap_sorted = cell_std[std_rows][:, std_cols]

    im0 = ax_mean_hm.imshow(
        mean_heatmap_sorted.numpy(),
        aspect="auto",
        interpolation="nearest",
        cmap="coolwarm",
        vmin=mean_hm_lo,
        vmax=mean_hm_hi,
    )

    ax_mean_hm.set_title(
        f"mean heatmap\n"
        f"data [{cell_mean.min().item():.3f}, {cell_mean.max().item():.3f}]   "
        f"color [{mean_hm_lo:.3f}, {mean_hm_hi:.3f}]",
        fontsize=10,
    )

    ax_mean_hm.set_xlabel("Embedding dimension (sorted)")
    ax_mean_hm.set_ylabel("Latent token (sorted)")

    im1 = ax_std_hm.imshow(
        std_heatmap_sorted.numpy(),
        aspect="auto",
        interpolation="nearest",
        cmap="coolwarm",
        vmin=std_hm_lo,
        vmax=std_hm_hi,
    )

    ax_std_hm.set_title(
        f"std heatmap\n"
        f"data [{cell_std.min().item():.3f}, {cell_std.max().item():.3f}]   "
        f"color [{std_hm_lo:.3f}, {std_hm_hi:.3f}]",
        fontsize=10,
    )

    ax_std_hm.set_xlabel("Embedding dimension (sorted)")
    ax_std_hm.set_ylabel("Latent token (sorted)")

    # ------------------------------------------------------------
    # COLORBARS
    # ------------------------------------------------------------

    cbar0 = fig.colorbar(
        im0,
        ax=ax_mean_hm,
        fraction=0.046,
        pad=0.02,
    )
    cbar0.set_label("Mean")

    cbar1 = fig.colorbar(
        im1,
        ax=ax_std_hm,
        fraction=0.046,
        pad=0.02,
    )
    cbar1.set_label("Std")

    # ------------------------------------------------------------
    # TITLE
    # ------------------------------------------------------------

    sign_label = "magnitude" if sort_abs else "signed"

    fig.suptitle(
        f"Latent stats dashboard — B={B}, L={L}, D={D}\n"
        f"sort={sort_by}, {sign_label}"
        f"   |   bars=actual"
        f"   |   green=randn expected"
        f"   |   yellow=randn 5–95%"
        f"   |   heatmaps={heatmap_percentile:g}% robust scale",
        fontsize=13,
    )

    plt.show()

In [ ]:
render_reconstruction(masked=False, split='train');

In [ ]:
render_reconstruction(masked=True);

In [ ]:
import matplotlib.pyplot as plt

enc, dec = encoder, decoder
enc_training, dec_training = enc.training, dec.training

try:
    enc.eval()
    dec.eval()

    with torch.no_grad():
        original = images[:4]
        flipped = original.flip(-2)  # Flip height, not width.

        with torch.autocast(
            device_type=original.device.type,
            dtype=torch.bfloat16,
            enabled=original.is_cuda,
        ):
            recon = dec(enc(original))
            flipped_recon = dec(enc(flipped))

finally:
    enc.train(enc_training)
    dec.train(dec_training)

panels = [original, recon, flipped, flipped_recon]
titles = ["Input", "Reconstruction", "Flipped input", "Flipped reconstruction"]

fig, axes = plt.subplots(
    len(original), 4, figsize=(12, 3 * len(original)), squeeze=False
)

for col, (batch, title) in enumerate(zip(panels, titles)):
    batch = batch.detach().float().cpu()
    for row, img in enumerate(batch):
        # Assumes RGB values in [0, 1].
        axes[row, col].imshow(
            img.permute(1, 2, 0).clamp(0, 1),
            interpolation="nearest",
        )
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

with torch.no_grad():
    idx = torch.randperm(len(images), device=images.device)[:6]
    originals = images[idx]
    grid = make_crop_grid(
        originals, crop_min=crop_min, crop_max=crop_max
    )
    crops = crop_scale(originals, grid)

originals = originals.detach().float().cpu()
crops = crops.detach().float().cpu()

fig, axes = plt.subplots(
    len(crops), 2, figsize=(7, 3 * len(crops)), squeeze=False
)

for i in range(len(crops)):
    for ax, img, title in zip(
        axes[i], [originals[i], crops[i]], ["Original", "Random crop"]
    ):
        ax.imshow(
            img.permute(1, 2, 0).clamp(0, 1),
            interpolation="nearest",  # No extra smoothing by the plot.
        )
        ax.set_title(title)
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# probe: reconstruction robustness to latent perturbation

@torch.no_grad()
def plot_z_robustness():

    features, _ = next(iter(train_dataloader))
    features = features[:8].to(device)

    z = encoder(features)
    z_std = z.std()

    noise_fracs = [
        0.00,
        0.10,
        0.20,
        0.50,
        0.75,
        0.85,
        0.95,
    ]

    noise = torch.randn_like(z)

    recons = []
    losses = []

    for frac in noise_fracs:

        z_noisy = (1-frac)*z + frac*noise

        x_hat = decoder(z_noisy)

        recons.append(x_hat.cpu())

        losses.append(
            F.mse_loss(
                x_hat.float(),
                features.float(),
            ).item()
        )

    # -------------------------
    # image row
    # -------------------------

    fig, axes = plt.subplots(
        1,
        len(noise_fracs),
        figsize=(2 * len(noise_fracs), 2.3),
    )

    for i, frac in enumerate(noise_fracs):

        img = (
            recons[i][0]
            .float()
            .permute(1, 2, 0)
            .clamp(0, 1)
        )

        axes[i].imshow(img)
        axes[i].set_title(f"noise {frac:.2f}")
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

    # -------------------------
    # robustness curve
    # -------------------------

    plt.figure(figsize=(7, 4))

    plt.plot(
        noise_fracs,
        losses,
        marker="o",
    )

    plt.xlabel("Latent noise / z std")
    plt.ylabel("Reconstruction MSE")
    plt.title("Latent robustness")

    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_z_robustness()

In [ ]:
def show_token_shuffles(n_shuffles=4):
    images, _ = next(iter(test_dataloader))
    idx = torch.randint(len(images), ()).item()
    images = images[idx:idx + 1].to(device)

    with torch.no_grad():
        z = encoder(images)

        reconstructions = [decoder(z)[0]]
        for _ in range(n_shuffles):
            perm = torch.randperm(z.shape[1], device=z.device)
            reconstructions.append(decoder(z[:, perm])[0])

    fig, axes = plt.subplots(
        1, n_shuffles + 1,
        figsize=(4 * (n_shuffles + 1), 4),
        squeeze=False,
    )

    titles = ["Original token order"] + [
        f"Shuffle {i + 1}" for i in range(n_shuffles)
    ]

    for ax, img, title in zip(axes.flat, reconstructions, titles):
        ax.imshow(img.detach().float().cpu().permute(1, 2, 0).clamp(0, 1))
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
show_token_shuffles()

In [ ]:
# interpolate two images
@torch.no_grad()
def interpolate_two_images(masked=False):

    features, _ = next(iter(train_dataloader))
    features = features[:2].to(device)

    if masked:
        mask, keep_indices = make_mask(
            features,
            mask_ratio=encoder_patch_drop_ratio,
            return_keep_indices=True,
        )
        z = encoder(features, keep_indices=keep_indices)
    else:
        z = encoder(features)

    z1, z2 = z.unbind(0)

    weights = torch.linspace(
        0,
        1,
        8,
        device=device,
    )

    rho = F.cosine_similarity(
        z1.flatten(),
        z2.flatten(),
        dim=0,
    )

    z_interpolated = torch.stack([
        torch.lerp(z1, z2, a)
        / torch.sqrt(
            (1 - a) ** 2
            + a ** 2
            + 2 * a * (1 - a) * rho
        )
        for a in weights
    ])

    plot_constructions(
        z_interpolated,
        patch_rows,
        patch_cols,
    )

In [ ]:
interpolate_two_images(masked=False)

In [ ]:
interpolate_two_images(masked=True)

In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def measure_view_invariance(dataloader, n_batches=20):

    mse_vals = []
    cos_vals = []
    rel_rmse_vals = []
    random_mse_vals = []

    for i, (images, *_) in enumerate(dataloader):
        if i >= n_batches:
            break

        images = images.to(device)

        # same images, clean vs masked
        mask, keep_indices = make_mask(
            images,
            mask_ratio=encoder_patch_drop_ratio,
            return_keep_indices=True,
        )

        z_clean = encoder(images)
        z_masked = encoder(images, keep_indices=keep_indices)

        # absolute displacement
        mse = (z_clean - z_masked).square().mean()

        # directional similarity
        cos = F.cosine_similarity(
            z_clean.flatten(1),
            z_masked.flatten(1),
            dim=1,
        ).mean()

        # displacement relative to latent amplitude
        rmse = mse.sqrt()
        z_rms = z_clean.square().mean().sqrt()
        rel_rmse = rmse / z_rms

        # baseline: unrelated clean images
        z_other = z_clean.roll(1, dims=0)
        random_mse = (z_clean - z_other).square().mean()

        mse_vals.append(mse)
        cos_vals.append(cos)
        rel_rmse_vals.append(rel_rmse)
        random_mse_vals.append(random_mse)

    mse = torch.stack(mse_vals).mean().item()
    cos = torch.stack(cos_vals).mean().item()
    rel_rmse = torch.stack(rel_rmse_vals).mean().item()
    random_mse = torch.stack(random_mse_vals).mean().item()

    print(f"clean ↔ masked MSE:     {mse:.6f}")
    print(f"clean ↔ masked cosine:  {cos:.4f}")
    print(f"relative RMSE:          {rel_rmse:.4f}")
    print(f"random-image MSE:       {random_mse:.6f}")
    print(f"invariance / random:    {mse / random_mse:.4f}")

In [ ]:
measure_view_invariance(test_dataloader)

In [ ]:
train_iter = iter(train_dataloader)

features, labels = next(train_iter)
features = features.to(device)

z = encoder(features)

print(z.shape)

plot_z_dashboard(
    z,
    n_ref=256,
    sort_by="individual",
    sort_abs=False,
)

In [ ]:
# latent correlation dashboard: plot_z_correlation_dashboard

import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def plot_z_correlation_dashboard(z):
    """
    z: (B, L, D)

    Channel correlation:
        observations = B * L
        variables    = D channels

    Token correlation:
        observations = B * D
        variables    = L tokens
    """

    z = z.detach().float()
    B, L, D = z.shape

    # --------------------------------------------------
    # CHANNEL <-> CHANNEL CORRELATION
    # pool over batch + tokens
    # --------------------------------------------------
    channel_samples = z.reshape(B * L, D)
    channel_corr = torch.corrcoef(channel_samples.T)

    channel_mask = ~torch.eye(
        D,
        dtype=torch.bool,
        device=z.device,
    )

    channel_offdiag = channel_corr[channel_mask]
    channel_abs_sorted = channel_offdiag.abs().sort(descending=True).values

    # --------------------------------------------------
    # TOKEN <-> TOKEN CORRELATION
    # pool over batch + channels
    # --------------------------------------------------
    token_samples = (
        z.permute(0, 2, 1)
        .reshape(B * D, L)
    )

    token_corr = torch.corrcoef(token_samples.T)

    token_mask = ~torch.eye(
        L,
        dtype=torch.bool,
        device=z.device,
    )

    token_offdiag = token_corr[token_mask]
    token_abs_sorted = token_offdiag.abs().sort(descending=True).values

    # --------------------------------------------------
    # SUMMARY STATS
    # --------------------------------------------------
    ch_mean = channel_offdiag.abs().mean().item()
    ch_median = channel_offdiag.abs().median().item()
    ch_max = channel_offdiag.abs().max().item()

    tok_mean = token_offdiag.abs().mean().item()
    tok_median = token_offdiag.abs().median().item()
    tok_max = token_offdiag.abs().max().item()

    # --------------------------------------------------
    # PLOT
    # --------------------------------------------------
    fig = plt.figure(
        figsize=(16, 9),
        layout="constrained",
    )

    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=(3, 1),
    )

    # CHANNEL HEATMAP
    ax_ch = fig.add_subplot(gs[0, 0])

    im_ch = ax_ch.imshow(
        channel_corr.cpu(),
        vmin=-1,
        vmax=1,
        cmap="coolwarm",
    )

    ax_ch.set_title(
        f"channel correlation\n"
        f"mean |r|={ch_mean:.3f}   "
        f"median={ch_median:.3f}   "
        f"max={ch_max:.3f}"
    )

    ax_ch.set_xlabel("embedding dimension")
    ax_ch.set_ylabel("embedding dimension")

    fig.colorbar(
        im_ch,
        ax=ax_ch,
        label="correlation",
        shrink=0.85,
    )

    # TOKEN HEATMAP
    ax_tok = fig.add_subplot(gs[0, 1])

    im_tok = ax_tok.imshow(
        token_corr.cpu(),
        vmin=-1,
        vmax=1,
        cmap="coolwarm",
    )

    ax_tok.set_title(
        f"token correlation\n"
        f"mean |r|={tok_mean:.3f}   "
        f"median={tok_median:.3f}   "
        f"max={tok_max:.3f}"
    )

    ax_tok.set_xlabel("latent token")
    ax_tok.set_ylabel("latent token")

    fig.colorbar(
        im_tok,
        ax=ax_tok,
        label="correlation",
        shrink=0.85,
    )

    # SORTED CHANNEL CORRELATIONS
    ax_ch_sort = fig.add_subplot(gs[1, 0])

    ax_ch_sort.plot(
        channel_abs_sorted.cpu()
    )

    ax_ch_sort.axhline(
        ch_mean,
        linestyle="--",
        label=f"mean = {ch_mean:.3f}",
    )

    ax_ch_sort.set_title("sorted |channel correlation|")
    ax_ch_sort.set_xlabel("channel pair")
    ax_ch_sort.set_ylabel("|r|")
    ax_ch_sort.set_ylim(0, 1)
    ax_ch_sort.legend()

    # SORTED TOKEN CORRELATIONS
    ax_tok_sort = fig.add_subplot(gs[1, 1])

    ax_tok_sort.plot(
        token_abs_sorted.cpu()
    )

    ax_tok_sort.axhline(
        tok_mean,
        linestyle="--",
        label=f"mean = {tok_mean:.3f}",
    )

    ax_tok_sort.set_title("sorted |token correlation|")
    ax_tok_sort.set_xlabel("token pair")
    ax_tok_sort.set_ylabel("|r|")
    ax_tok_sort.set_ylim(0, 1)
    ax_tok_sort.legend()

    fig.suptitle(
        f"Latent correlation dashboard — "
        f"B={B}, L={L}, D={D}",
        fontsize=15,
    )

    plt.show()

    return {
        "channel_corr": channel_corr.cpu(),
        "token_corr": token_corr.cpu(),

        "channel_mean_abs_corr": ch_mean,
        "channel_median_abs_corr": ch_median,
        "channel_max_abs_corr": ch_max,

        "token_mean_abs_corr": tok_mean,
        "token_median_abs_corr": tok_median,
        "token_max_abs_corr": tok_max,
    }

In [ ]:
plot_z_correlation_dashboard(z);

In [ ]:
# Full flattened-latent correlation dashboard
# Matches current SWD space: (B, T*C)

import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def plot_flat_latent_correlation(z):
    """
    z: (B, T, C)

    Measures correlation between every native (token, channel)
    coordinate across B examples.
    """

    z = z.detach().float()

    B, T, C = z.shape
    E = T * C

    # Same flattening as current SWD
    z_flat = z.flatten(1)  # (B, E)

    # Correlation across examples
    x = z_flat - z_flat.mean(dim=0, keepdim=True)
    x = x / x.std(dim=0, keepdim=True).clamp_min(1e-8)

    corr = (x.T @ x) / (B - 1)

    eye = torch.eye(
        E,
        dtype=torch.bool,
        device=z.device,
    )
    off_diag = ~eye

    abs_corr = corr[off_diag].abs()
    sorted_abs = abs_corr.sort(descending=True).values

    mean_abs = abs_corr.mean().item()
    median_abs = abs_corr.median().item()
    max_abs = abs_corr.max().item()

    print(f"B={B} | T={T} | C={C} | flattened dims={E}")
    print(
        f"off-diagonal |r|: "
        f"mean={mean_abs:.4f} | "
        f"median={median_abs:.4f} | "
        f"max={max_abs:.4f}"
    )

    # Full correlation matrix
    plt.figure(figsize=(18, 16))
    plt.imshow(
        corr.cpu(),
        vmin=-1,
        vmax=1,
        cmap="coolwarm",
        interpolation="nearest",
    )
    plt.colorbar(label="correlation")
    plt.title(
        f"Full latent coordinate correlation\n"
        f"B={B}, E={E} | mean |r|={mean_abs:.3f}"
    )
    plt.xlabel("flattened token × channel dim")
    plt.ylabel("flattened token × channel dim")
    plt.show()

    # Sorted correlation strengths
    plt.figure(figsize=(12, 5))
    plt.plot(sorted_abs.cpu())

    plt.axhline(
        mean_abs,
        linestyle="--",
        label=f"mean = {mean_abs:.3f}",
    )

    plt.xlabel("coordinate pair, sorted")
    plt.ylabel("|correlation|")
    plt.title("Sorted full-latent coordinate correlations")
    plt.legend()
    plt.show()

    return {
        "corr": corr.cpu(),
        "mean_abs_corr": mean_abs,
        "median_abs_corr": median_abs,
        "max_abs_corr": max_abs,
    }



In [ ]:
# flat_corr_stats = plot_flat_latent_correlation(z)

In [ ]:
# cosine similarity distribution between unrelated encoded images

import torch
import matplotlib.pyplot as plt

encoder.eval()

with torch.no_grad():
    features, _ = next(iter(train_dataloader))
    features = features.to(device)

    z= encoder(features)      # [B, T, C]

    # treat each image's whole latent as one vector
    z_flat = z.flatten(1)               # [B, T*C]
    z_flat = torch.nn.functional.normalize(z_flat, dim=1)

    # cosine similarity of every image pair
    cosine_matrix = z_flat @ z_flat.T   # [B, B]

    # keep each unique pair once, excluding self-similarity
    mask = torch.triu(
        torch.ones_like(cosine_matrix, dtype=torch.bool),
        diagonal=1
    )

    cosines = cosine_matrix[mask].cpu()


print(f"pairs:  {len(cosines):,}")
print(f"mean:   {cosines.mean():.3f}")
print(f"std:    {cosines.std():.3f}")
print(f"min:    {cosines.min():.3f}")
print(f"median: {cosines.median():.3f}")
print(f"max:    {cosines.max():.3f}")


plt.figure(figsize=(8, 4))
plt.hist(cosines.numpy(), bins=40)
plt.axvline(cosines.mean(), linestyle="--", label=f"mean = {cosines.mean():.3f}")
plt.xlabel("cosine similarity")
plt.ylabel("pair count")
plt.title("Cosine similarity between image latents")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def scan_first_attention(images):
    E = getattr(encoder, "_orig_mod", encoder)
    D = getattr(decoder, "_orig_mod", decoder)
    attn = D.core_transformer.blocks[0].MHA
    x = images[:16].to(next(E.parameters()).device)
    captured = {}

    def capture(module, args, output):
        captured["input"] = args[0].detach()
        captured["rope"] = args[1]
        captured["output"] = output.detach()

    handle = attn.register_forward_hook(capture)

    try:
        with torch.random.fork_rng(
            devices=[x.device.index] if x.is_cuda else []
        ):
            with torch.no_grad(), torch.autocast(
                device_type=x.device.type,
                dtype=torch.bfloat16,
                enabled=x.is_cuda,
            ):
                z = E(x)
                D(z)

                h = attn.norm(captured["input"])
                B, T, _ = h.shape
                H, S = attn.n_heads, attn.head_size
                q, k, v = attn.QKV(h).chunk(3, dim=-1)

                def heads(t):
                    return t.reshape(B, T, H, S).transpose(1, 2)

                q, k = apply_2d_rope(
                    heads(q), heads(k), captured["rope"]
                )
                v = heads(v)
    finally:
        handle.remove()

    with torch.no_grad(), torch.autocast(
        device_type=x.device.type, enabled=False
    ):
        P = captured["rope"][0].shape[-2]
        L = T - P
        Z = z.shape[1]

        # Only patch queries; all keys. Compute scores in FP32.
        scores = q[:, :, L:].float() @ k.float().transpose(-2, -1)
        scores = scores / S**0.5
        a = scores.softmax(dim=-1)
        v = v.float()

        groups = [
            ("encoded", 0, Z),
            ("slots", Z, L),
            ("patches", L, T),
        ]
        mass = [
            a[..., start:end].sum(-1).mean((0, 2))
            for _, start, end in groups
        ]
        peak = a.max(-1).values.mean((0, 2))
        entropy = -(a * a.clamp_min(1e-30).log()).sum(-1)
        effective = entropy.exp().mean((0, 2))

        print("Patch-query attention, averaged over images and positions")
        print(f"{'head':>5} {'encoded':>10} {'slots':>10} "
              f"{'patches':>10} {'max prob':>10} {'eff keys':>10}")
        for i in range(H):
            values = [m[i].item() for m in mass]
            values += [peak[i].item(), effective[i].item()]
            print(f"{i:5d}" + "".join(f"{v:10.3f}" for v in values))

        print("\nUniform-attention reference:")
        print(", ".join(
            f"{name}={(end-start)/T:.1%}"
            for name, start, end in groups
        ))

        W = attn.out_proj.weight.detach().float()
        parts = []

        def stats(name, t):
            rms = t.square().mean().sqrt().item()
            image_sd = t.var(0, unbiased=False).mean().sqrt().item()
            token_sd = t.var(1, unbiased=False).mean().sqrt().item()
            print(f"{name:20} {rms:11.4g} "
                  f"{image_sd:11.4g} {token_sd:11.4g}")

        print("\nContributions after output projection")
        print(f"{'source':20} {'RMS':>11} "
              f"{'image SD':>11} {'patch SD':>11}")

        for name, start, end in groups:
            contribution = a[..., start:end] @ v[:, :, start:end]
            contribution = contribution.transpose(1, 2).reshape(B, P, H*S)
            contribution = F.linear(contribution, W, bias=None)
            parts.append(contribution)
            stats(name, contribution)

        bias = torch.zeros_like(parts[0])
        if attn.out_proj.bias is not None:
            bias = bias + attn.out_proj.bias.detach().float()
        stats("projection bias", bias)

        reconstructed = sum(parts) + bias
        actual = (
            captured["output"][:, L:].float()
            - captured["input"][:, L:].float()
        )
        stats("summed update", reconstructed)
        stats("actual update", actual)

        relative_error = (
            (reconstructed - actual).square().mean().sqrt()
            / actual.square().mean().sqrt().clamp_min(1e-12)
        )
        print(f"\nFP32 reconstruction relative error: {relative_error.item():.3%}")

scan_first_attention(images)

In [ ]:
# Check whether a few directions dominate image-to-image latent variation
def latent_pca():
    import torch
    import matplotlib.pyplot as plt
    
    with torch.no_grad():
        x = z.detach().flatten(1).float()       # [B, T*C], not normalized
        x = x - x.mean(dim=0, keepdim=True)     # remove the shared mean vector
    
        if min(x.shape) < 2:
            raise ValueError("Need at least two images and two latent coordinates.")
    
        u, s, _ = torch.linalg.svd(x, full_matrices=False)
        energy = s.square()
    
        if energy.sum().item() == 0:
            raise ValueError("All image latents are identical in this batch.")
    
        # Centering leaves at most B-1 independent directions
        n = min(x.shape[0] - 1, x.shape[1])
        share = (energy[:n] / energy.sum()).cpu()
        cumulative = share.cumsum(0)
        scores = (u[:, :2] * s[:2]).cpu()       # image positions along PC1 and PC2
    
        effective_dims = 1 / share.square().sum()
        dims_90 = torch.searchsorted(cumulative, 0.90).item() + 1
    
    print(f"Images: {x.shape[0]} | Flattened dimensions: {x.shape[1]:,}")
    print(f"Top direction: {share[0]:.1%} of centered variation")
    print(f"Top 2 directions: {share[:2].sum():.1%}")
    print(f"Top 5 directions: {share[:5].sum():.1%}")
    print(f"Directions needed for 90%: {dims_90}")
    print(f"Effective dimensions (participation ratio): {effective_dims:.1f}")
    print(f"Batch can reveal at most {n} independent directions.")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    k = min(20, n)
    axes[0].bar(range(1, k + 1), 100 * share[:k].numpy())
    axes[0].set(
        title="Variation carried by each direction",
        xlabel="Principal direction",
        ylabel="Share of variation (%)",
    )
    
    axes[1].plot(range(1, n + 1), 100 * cumulative.numpy())
    axes[1].axhline(90, color="gray", linestyle="--", label="90%")
    axes[1].set(
        title="Cumulative variation",
        xlabel="Number of directions",
        ylabel="Variation captured (%)",
        ylim=(0, 102),
    )
    axes[1].legend()
    
    axes[2].scatter(scores[:, 0], scores[:, 1], s=28, alpha=0.65)
    axes[2].axhline(0, color="gray", linewidth=0.7)
    axes[2].axvline(0, color="gray", linewidth=0.7)
    axes[2].set(
        title="Each dot is one image",
        xlabel=f"PC1 ({share[0]:.1%})",
        ylabel=f"PC2 ({(energy[1] / energy.sum()).item():.1%})",
    )
    axes[2].set_aspect("equal", adjustable="box")
    
    for ax in axes:
        ax.grid(alpha=0.2)
    
    plt.tight_layout()
    plt.show()

In [ ]:
latent_pca()

In [ ]:
#def latent ablations
import math
import torch
import matplotlib.pyplot as plt

@torch.no_grad()
def probe_latent_ablations(
    image_index=0,
    n_svd=8,
    token_counts=(1, 2, 4, 8, 12, 16, 20, 24),
    channel_counts=(1, 2, 4, 8, 16, 32, 48, 64),
    seed=42,
):
    if n_svd < 1:
        raise ValueError("n_svd must be >= 1")

    modes = [
        (module, module.training)
        for model in (encoder, decoder)
        for module in model.modules()
    ]
    probe_device = next(encoder.parameters()).device

    try:
        encoder.eval()
        decoder.eval()

        # Grab one source image + latent from first batch.
        for batch_i, (images, _) in enumerate(train_dataloader):
            images = images.to(probe_device)
            z = encoder(images)

            if not 0 <= image_index < len(images):
                raise ValueError("image_index is outside the first batch.")

            source_image = images[image_index].float().cpu()
            source_z = z[image_index:image_index + 1].float()   # (1, T, D)
            break
        else:
            raise ValueError("train_dataloader is empty.")

        z = source_z
        _, T, D = z.shape
        target_norm = z.norm()

        # Per-image SVD on the latent matrix z[0] with shape (T, D).
        z2d = z[0]
        U, S, Vh = torch.linalg.svd(z2d, full_matrices=False)

        n_svd = min(n_svd, len(S))
        energy_fraction = S.square() / S.square().sum().clamp_min(1e-12)

        # Rank-1 components, each shaped (1, T, D)
        components = []
        for i in range(n_svd):
            comp = (
                S[i]
                * U[:, i:i+1]
                * Vh[i:i+1, :]
            )
            components.append(comp.unsqueeze(0))
        components = torch.cat(components, dim=0)          # (n_svd, T, D)
        cumulative = components.cumsum(dim=0)              # top-k reconstruction

        def norm_match(values):
            return values * (
                target_norm / values.norm().clamp_min(1e-12)
            )

        rng = torch.Generator(device="cpu").manual_seed(seed)
        token_order = torch.randperm(T, generator=rng).to(probe_device)
        channel_order = torch.randperm(D, generator=rng).to(probe_device)

        def valid_counts(counts, size):
            return sorted({
                min(int(k), size)
                for k in counts
                if int(k) > 0
            })

        token_counts = valid_counts(token_counts, T)
        channel_counts = valid_counts(channel_counts, D)

        def decode(probe_z):
            return decoder(
                probe_z.to(source_z.dtype)
            )[0].float().cpu()

        def select(values, axis, indices):
            selected = torch.zeros_like(values)
            if axis == "Token":
                selected[:, indices, :] = values[:, indices, :]
            else:
                selected[:, :, indices] = values[:, :, indices]
            return selected

        def comp_label(i):
            return f"C{i + 1}"

        full_recon = decode(z)
        zero_recon = decode(torch.zeros_like(z))
        
        sections = {}

        def add_section(name, probes):
            sections[name] = [
                (title, decode(probe_z))
                for title, probe_z in probes
            ]

        # SVD ablations: remove top-k components from raw z, no renorm.
        add_section(
            "SVD ablate: cumulative",
            [
                (
                    f"Remove {comp_label(i)}",
                    z - cumulative[i:i + 1],
                )
                for i in range(n_svd)
            ],
        )


        # SVD only: individual, renorm to full latent norm.
        add_section(
            "SVD only: individual | 1× norm",
            [
                (
                    f"{comp_label(i)} only\n"
                    f"{energy_fraction[i].item():.1%} energy",
                    norm_match(components[i:i + 1]),
                )
                for i in range(n_svd)
            ],
        )

        for axis, order, counts in [
            ("Token", token_order, token_counts),
            ("Channel", channel_order, channel_counts),
        ]:
            # Natural ablation: zero selected entries, no renorm.
            add_section(
                f"{axis} ablate: cumulative",
                [
                    (
                        f"Remove {k}/{len(order)}",
                        z - select(z, axis, order[:k]),
                    )
                    for k in counts
                ],
            )

            # Keep only selected entries, renorm to full latent norm.

            add_section(
                f"{axis} only: individual | 1× norm",
                [
                    (
                        f"{axis} {i}",
                        norm_match(select(z, axis, [i])),
                    )
                    for i in range(len(order))
                ],
            )

    finally:
        for module, was_training in modes:
            module.train(was_training)

    def show_grid(title, items, ncols=8, cell_size=2):
        if not items:
            return

        ncols = min(ncols, len(items))
        nrows = math.ceil(len(items) / ncols)

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(cell_size * ncols, cell_size * nrows + 0.7),
            squeeze=False,
            layout="constrained",
        )

        for ax, (label, img) in zip(axes.flat, items):
            ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
            ax.set_title(label, fontsize=9)

        for ax in axes.flat:
            ax.axis("off")

        fig.suptitle(title)
        plt.show()

    show_grid(
        "Reference",
        [
            ("Input", source_image),
            ("Full reconstruction", full_recon),
            ("Zero latent", zero_recon),
        ],
        ncols=3,
        cell_size=2.5,
    )

    for name, items in sections.items():
        show_grid(name, items)

    print(f"Single-image SVD | latent: {T} tokens × {D} channels")
    print(
        f"First {n_svd} components explain "
        f"{energy_fraction[:n_svd].sum().item():.1%} of latent energy"
    )
    print("Token subset order:", token_order.cpu().tolist())
    print("Channel subset order:", channel_order.cpu().tolist())
    print("ABLATE: selected information is zeroed with no renorm.")
    print("ONLY: selected information is kept and scaled to full latent norm.")

    return {
        "input": source_image,
        "full_recon": full_recon,
        "zero_recon": zero_recon,
        "sections": sections,
        "z": z.cpu(),
        "svd_u": U[:, :n_svd].cpu(),
        "svd_s": S[:n_svd].cpu(),
        "svd_vh": Vh[:n_svd].cpu(),
        "svd_components": components.cpu(),
        "energy_fraction": energy_fraction[:n_svd].cpu(),
        "token_order": token_order.cpu(),
        "channel_order": channel_order.cpu(),
    }



In [ ]:
ablation_results = probe_latent_ablations()

In [ ]:
render_reconstruction(masked=False);

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
####################### STOP - SAVING/OVERWRITING BELOW #############################

In [ ]:
################ EXPORT REFINED EXPORTED AE ################

import os
from pathlib import Path
import torch


def export_refined_ae(
    encoder,
    decoder,
    example_images,
    name,
    save_root=None,
):
    if save_root is None:
        save_root = os.environ["ML_MODELS"]

    save_dir = Path(save_root) / name
    save_dir.mkdir(parents=True, exist_ok=True)

    # >1 example so batch dimension can be dynamic
    example_images = example_images[:2]

    batch_dim = torch.export.Dim(
        "batch",
        min=1,
        max=8192,
    )

    # --------------------------------
    # Get example latent
    # --------------------------------
    with torch.no_grad():
        z = encoder(example_images)

    # --------------------------------
    # Encoder
    #
    # Export:
    #     image -> z
    # --------------------------------
    encoder_ep = torch.export.export(
        encoder,
        (example_images,),
        dynamic_shapes=(
            {0: batch_dim},
        ),
        strict=False,
    )

    # --------------------------------
    # Decoder
    #
    # Export:
    #     z -> image
    # --------------------------------
    decoder_ep = torch.export.export(
        decoder,
        (z,),
        dynamic_shapes=(
            {0: batch_dim},
        ),
        strict=False,
    )

    # --------------------------------
    # Save
    # --------------------------------
    torch.export.save(
        encoder_ep,
        save_dir / "encoder.pt2",
    )

    torch.export.save(
        decoder_ep,
        save_dir / "decoder.pt2",
    )

    print(f"Exported refined AE to {save_dir}")

In [ ]:
## EXPORT  AE ##

ae_name = "ae_11M_self-teach_color_drop_10p_241e"

images, _ = next(iter(train_dataloader))
images = images[:2].to(device)

export_refined_ae(
    encoder,
    decoder,
    images,
    ae_name,
)

In [ ]:
import os
import torch

name = '.pt' # pre-decoder refinement

In [ ]:
# load models

import os
from pathlib import Path

checkpoint_path = (
    Path(os.environ["ML_CHECKPOINTS"])
    / "ae"
    / name
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)

encoder.load_state_dict(checkpoint["encoder"])
decoder.load_state_dict(checkpoint["decoder"])

print(f"Loaded checkpoint from {checkpoint_path}")

In [ ]:
# save models

import os
from pathlib import Path

checkpoint_dir = Path(os.environ["ML_CHECKPOINTS"]) / "ae"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / name

torch.save(
    {
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
    },
    checkpoint_path,
)

print(f"Saved checkpoint to {checkpoint_path}")

In [ ]:
# sync models + checkpoints + training_runs   spark <---> pc

import os
import socket
import subprocess
from pathlib import Path


def sync_models():

    hostname = socket.gethostname()

    # Explicit paths. Do NOT infer everything from ML_MODELS.parent.
    if hostname == "gpu-box":

        pairs = {
            "models": (
                Path(os.environ["ML_MODELS"]),
                "pc:<home>/jupyter/models/",
            ),

            "checkpoints": (
                Path(os.environ["ML_CHECKPOINTS"]),
                "pc:<home>/jupyter/checkpoints/",
            ),

            "training_runs": (
                Path("<home>/projects/Jupyter_AI/training_runs"),
                "pc:<home>/jupyter/training_runs/",
            ),
        }

    elif hostname == "pc":

        pairs = {
            "models": (
                Path(os.environ["ML_MODELS"]),
                "gpu-box:<home>/jupyterlab/Jupyter_AI/models/",
            ),

            "checkpoints": (
                Path(os.environ["ML_CHECKPOINTS"]),
                "gpu-box:<home>/projects/Jupyter_AI/checkpoints/",
            ),

            "training_runs": (
                Path("<home>/jupyter/training_runs"),
                "gpu-box:<home>/projects/Jupyter_AI/training_runs/",
            ),
        }

    else:
        raise RuntimeError(f"Unknown host: {hostname}")

    rsync_args = [
        "rsync",
        "-a",
        "--update",
        "--partial",
        "--itemize-changes",
    ]

    for name, (local_dir, remote_dir) in pairs.items():

        local_dir.mkdir(parents=True, exist_ok=True)

        print(f"\nSyncing {name}...")
        print(f"  local:  {local_dir}")
        print(f"  remote: {remote_dir}")

        # Send newer local files
        subprocess.run(
            rsync_args + [
                f"{local_dir}/",
                remote_dir,
            ],
            check=True,
        )

        # Fetch newer remote files
        subprocess.run(
            rsync_args + [
                remote_dir,
                f"{local_dir}/",
            ],
            check=True,
        )

    print("\nModels + checkpoints + training runs synced.")

In [ ]:
sync_models()